# Self-Contained RAG Pipeline Notebook

This notebook replicates the full RAG backend as standalone, portable code.

**Pipeline**: Parse → Chunk → Enrich → Embed → Index → Search → Synthesize

**Supported formats**: PDF, DOCX, PPTX, CSV, XLSX

In [1]:
# Install dependencies
!pip install -q sentence-transformers faiss-cpu rank-bm25 openai aiosqlite pandas chardet numpy Pillow PyMuPDF python-docx python-pptx pydantic pydantic-settings httpx

# Optional: layout detection and Gemini
# !pip install -q doclayout-yolo huggingface_hub google-generativeai

import asyncio
import hashlib
import json
import logging
import pickle
import re
import time
from abc import ABC, abstractmethod
from collections import OrderedDict
from concurrent.futures import ThreadPoolExecutor
from contextlib import asynccontextmanager
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any, Literal, Optional, List, Tuple, Set

import aiosqlite
import chardet
import numpy as np
import pandas as pd
import fitz  # PyMuPDF

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")
logger = logging.getLogger("rag_pipeline")


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Configuration

In [2]:
@dataclass
class NotebookSettings:
    # Paths
    watch_folder: Path = Path("./documents")
    data_folder: Path = Path("./data")

    # LLM Configuration
    llm_provider: str = "openai"  # "openai" or "gemini"
    openai_api_key: str = ""
    openai_model: str = "gpt-4o-mini"
    gemini_api_key: str = ""
    gemini_model: str = "gemini-1.5-flash"
    enrichment_model: str = ""
    synthesis_model: str = ""

    # VLM Configuration
    vlm_provider: str = "ollama"
    use_vlm_extraction: bool = True
    vlm_fallback_ocr: bool = True
    ollama_base_url: str = "http://localhost:11434"
    ollama_model: str = "qwen3-vl:8b"
    ollama_timeout: float = 120.0
    huggingface_api_key: str = ""
    huggingface_vlm_model: str = "Qwen/Qwen2.5-VL-32B-Instruct:fireworks-ai"
    huggingface_timeout: float = 60.0

    # Embedding
    embedding_model: str = "all-mpnet-base-v2"
    embedding_dimension: int = 768

    # Chunking
    chunk_size: int = 1500
    chunk_overlap: int = 0
    max_chunk_size: int = 2500
    enable_semantic_chunking: bool = True
    semantic_similarity_threshold: float = 0.72
    enable_chunk_links: bool = True
    chunking_strategy: str = "hierarchical"
    csv_rows_per_chunk: int = 50
    csv_max_rows_to_index: int = 0

    # Cache
    cache_ttl_seconds: int = 3600
    cache_similarity_threshold: float = 0.92
    cache_max_entries: int = 2000

    # Search
    search_top_k: int = 10

    # Synthesis
    synthesis_max_tokens: int = 1500
    synthesis_temperature: float = 0.1
    synthesis_max_chunks: int = 10

    # RRF
    rrf_k_constant: int = 30

    # HyDE
    enable_hyde: bool = True
    hyde_num_hypotheticals: int = 1
    hyde_alpha: float = 0.5
    hyde_alpha_min: float = 0.3
    hyde_alpha_max: float = 0.6
    hyde_adaptive: bool = True

    # Re-ranking
    reranker_enabled: bool = False
    reranker_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    reranker_top_k: int = 20
    reranker_blend_ratio: float = 0.7

    # MMR
    mmr_enabled: bool = False
    mmr_lambda: float = 0.7

    # BM25
    bm25_expand_synonyms: bool = True
    bm25_min_token_length: int = 2

    # Query
    query_expand_synonyms: bool = True

    # Enrichment
    enrichment_batch_size: int = 5
    questions_per_chunk: int = 5

    # Multi-Vector Weights
    multi_vector_weights_main: float = 0.40
    multi_vector_weights_question: float = 0.20
    multi_vector_weights_bm25: float = 0.25
    multi_vector_weights_summary: float = 0.15

    # Database
    database_path: Path = field(default_factory=lambda: Path(":memory:"))

    def index_path(self, name: str) -> Path:
        return self.data_folder / name

    @property
    def bm25_index_path(self) -> Path:
        return self.index_path("bm25/index.pkl")

    @property
    def cache_path(self) -> Path:
        return self.index_path("cache/semantic_cache.pkl")

    @property
    def multi_vector_weights(self) -> dict:
        return {
            "main_vector": self.multi_vector_weights_main,
            "question_vector": self.multi_vector_weights_question,
            "bm25": self.multi_vector_weights_bm25,
            "summary_vector": self.multi_vector_weights_summary,
        }

    @property
    def weight_profile_factual(self) -> dict:
        return {"main_vector": 0.35, "bm25": 0.35, "question_vector": 0.20, "summary_vector": 0.10}

    @property
    def weight_profile_exploratory(self) -> dict:
        return {"main_vector": 0.45, "bm25": 0.15, "question_vector": 0.25, "summary_vector": 0.15}

    @property
    def weight_profile_comparative(self) -> dict:
        return {"main_vector": 0.35, "bm25": 0.20, "question_vector": 0.30, "summary_vector": 0.15}

    @property
    def weight_profile_aggregation(self) -> dict:
        return {"main_vector": 0.30, "bm25": 0.30, "question_vector": 0.20, "summary_vector": 0.20}

    def get_weight_profile(self, query_type: str) -> dict:
        profiles = {
            "factual": self.weight_profile_factual,
            "exploratory": self.weight_profile_exploratory,
            "comparative": self.weight_profile_comparative,
            "aggregation": self.weight_profile_aggregation,
        }
        return profiles.get(query_type, self.multi_vector_weights)


settings = NotebookSettings()

## Data Models

In [ ]:
from pydantic import BaseModel, Field

# --- Enrichment ---
@dataclass
class EnrichmentResult:
    document_type: str = "unknown"
    summary: str = ""
    entities: dict[str, Any] = field(default_factory=dict)
    key_topics: list[str] = field(default_factory=list)
    table_descriptions: list[str] = field(default_factory=list)
    success: bool = True
    error: str | None = None

# --- Parser models ---
@dataclass
class TableData:
    headers: list[str]
    rows: list[list[Any]]
    title: str | None = None
    sheet_name: str | None = None

@dataclass
class HeadingInfo:
    text: str
    level: int
    page: int | None = None

@dataclass
class ParseResult:
    text: str = ""
    headings: list[HeadingInfo] = field(default_factory=list)
    tables: list[TableData] = field(default_factory=list)
    paragraphs: list[str] = field(default_factory=list)
    metadata: dict[str, Any] = field(default_factory=dict)
    column_headers: list[str] | None = None
    column_types: dict[str, str] | None = None
    sample_rows: list[list[Any]] | None = None
    row_count: int | None = None
    sheet_names: list[str] | None = None
    numeric_stats: dict[str, dict[str, float]] | None = None
    dataframe: Any | None = None
    markdown: str | None = None

# --- Chunk models ---
class Chunk(BaseModel):
    id: str = Field(description="Unique chunk identifier")
    document_id: str = Field(description="Parent document ID")
    text: str = Field(description="Original chunk content")
    contextualized_text: str = Field(description="Enriched text with document context")
    content_type: Literal["paragraph", "table", "list", "summary", "schema", "heading", "title", "section_header", "figure", "row_batch"] = "paragraph"
    page: int | None = None
    sheet_name: str | None = None
    heading_path: str | None = None
    entities: dict[str, Any] = Field(default_factory=dict)
    chunk_index: int = 0
    parent_chunk_id: str | None = None
    hierarchy_level: int = 0
    bbox: list[float] | None = None
    layout_label: str | None = None
    is_semantic_boundary: bool = False
    semantic_similarity_prev: float | None = None
    prev_chunk_id: str | None = None
    next_chunk_id: str | None = None
    prev_chunk_summary: str | None = None
    next_chunk_summary: str | None = None

class ChunkMetadata(BaseModel):
    chunk_id: str
    title: str | None = None
    summary: str | None = None
    keywords: list[str] = Field(default_factory=list)
    entities: dict[str, list[str]] = Field(default_factory=dict)
    category: Literal["definition", "procedure", "data", "narrative", "example", "reference"] | None = None
    contextual_description: str | None = None
    temporal_context: str | None = None
    enriched_at: datetime | None = None

class ChunkQuestion(BaseModel):
    id: int | None = None
    chunk_id: str
    question: str
    vector_id: str | None = None

class VectorEmbedding(BaseModel):
    id: str
    chunk_id: str
    vector_type: Literal["main", "summary", "question"]
    source_text: str | None = None
    question_id: int | None = None

# --- Document models ---
class Document(BaseModel):
    id: str
    file_path: str
    file_name: str
    file_type: str
    file_hash: str
    detected_doc_type: str = "unknown"
    summary: str = ""
    entities: dict[str, Any] = Field(default_factory=dict)
    key_topics: list[str] = Field(default_factory=list)
    table_descriptions: list[str] = Field(default_factory=list)
    indexed_at: datetime = Field(default_factory=datetime.utcnow)
    sheet_names: list[str] | None = None
    column_schema: dict[str, str] | None = None
    row_count: int | None = None
    date_range: str | None = None
    processing_status: str = "pending"
    layout_data: str | None = None
    extracted_markdown: str | None = None
    reviewed_markdown: str | None = None
    page_count: int | None = None

class DocumentSummary(BaseModel):
    id: str
    file_name: str
    file_type: str
    detected_doc_type: str
    summary: str
    indexed_at: datetime

# --- Search models ---
class SourceInfo(BaseModel):
    document_id: str
    file_name: str
    file_type: str
    detected_doc_type: str
    chunks_used: int = 1
    chunk_ids: list[str] = Field(default_factory=list)

class SearchResultItem(BaseModel):
    document_id: str
    file_name: str
    file_type: str
    detected_doc_type: str
    chunk_text: str
    chunk_id: str
    score: float
    page: int | None = None
    sheet_name: str | None = None
    heading_path: str | None = None
    highlights: list[str] = Field(default_factory=list)
    entities: dict[str, Any] = Field(default_factory=dict)
    temporal_context: str | None = None
    chunk_title: str | None = None
    chunk_keywords: list[str] = Field(default_factory=list)

class SearchResponse(BaseModel):
    query: str
    results: list[SearchResultItem]
    answer: str | None = None
    total_results: int
    latency_ms: float
    cache_hit: bool = False
    response_tier: Literal["cache", "retrieval", "synthesis"] = "retrieval"
    sources: list[SourceInfo] = Field(default_factory=list)

class SearchRequest(BaseModel):
    query: str = Field(min_length=1, max_length=1000)
    filters: dict[str, Any] | None = None
    limit: int = Field(default=10, ge=1, le=100)
    mode: Literal["auto", "retrieval", "synthesis"] = "auto"

# --- Query intent ---
@dataclass
class QueryIntent:
    search_terms: str
    doc_type_filter: str | None = None
    entity_filters: dict[str, Any] = field(default_factory=dict)
    needs_synthesis: bool = False
    query_type: Literal["factual", "exploratory", "comparative", "aggregation"] = "factual"
    preferred_categories: list[str] = field(default_factory=list)

## Domain Knowledge

In [ ]:
# ---------------------------------------------------------------------------
# Insurance acronyms (merged superset)
# ---------------------------------------------------------------------------
INSURANCE_ACRONYMS: dict[str, str] = {
    # Certificate and documentation
    "coi": "certificate of insurance",
    "acord": "association for cooperative operations research and development",
    # Liability types
    "gl": "general liability",
    "cgl": "commercial general liability",
    "pl": "professional liability",
    "pol": "professional liability",
    "el": "employers liability",
    # Workers compensation
    "wc": "workers compensation",
    "wci": "workers compensation insurance",
    # Directors and officers
    "d&o": "directors and officers",
    "dno": "directors and officers",
    "do": "directors officers",
    # Errors and omissions
    "e&o": "errors and omissions",
    "eno": "errors and omissions",
    # Property and casualty
    "p&c": "property and casualty",
    "pc": "property casualty",
    # Employment practices
    "epli": "employment practices liability insurance",
    "epl": "employment practices liability",
    # Business types
    "bop": "business owners policy",
    "cpp": "commercial package policy",
    # Auto coverage
    "um": "uninsured motorist",
    "uim": "underinsured motorist",
    "pip": "personal injury protection",
    "bi": "bodily injury",
    "pd": "property damage",
    "comp": "comprehensive",
    "coll": "collision",
    # Other coverage types
    "cyb": "cyber liability",
    "cyber": "cyber liability",
    "aop": "all other perils",
    "bpp": "business personal property",
    "bii": "business income insurance",
    "ee": "extra expense",
    # Limits and deductibles
    "occ": "occurrence",
    "agg": "aggregate",
    "ded": "deductible",
    "sir": "self insured retention",
    # Process and documentation
    "fnol": "first notice of loss",
    "siu": "special investigations unit",
    "tpa": "third party administrator",
    "mga": "managing general agency",
    # Reinsurance
    "ri": "reinsurance",
    "xs": "excess",
    "qsr": "quota share reinsurance",
}

# ---------------------------------------------------------------------------
# Insurance-domain synonyms (used by keyword / BM25 index)
# ---------------------------------------------------------------------------
INSURANCE_SYNONYMS: dict[str, list[str]] = {
    "policy": ["coverage", "plan", "contract"],
    "premium": ["rate", "cost", "price", "payment"],
    "claim": ["loss", "incident", "occurrence"],
    "deductible": ["excess", "retention", "selfinsured"],
    "coverage": ["protection", "insurance", "policy"],
    "endorsement": ["rider", "amendment", "addendum"],
    "underwriting": ["riskassessment", "evaluation"],
    "insured": ["policyholder", "client", "customer"],
    "beneficiary": ["payee", "recipient"],
    "carrier": ["insurer", "company", "provider"],
    "agent": ["broker", "producer", "representative"],
    "renewal": ["extension", "continuation"],
    "cancellation": ["termination", "void", "cancel"],
    "exclusion": ["exception", "limitation"],
    "limit": ["maximum", "cap", "ceiling"],
    "liability": ["responsibility", "obligation"],
    "peril": ["risk", "hazard", "danger"],
    "sublimit": ["sublimitation", "internalimit"],
    "coinsurance": ["costsharing", "copay"],
    "indemnity": ["compensation", "reimbursement"],
}

# ---------------------------------------------------------------------------
# Common insurance-term variations / misspellings
# ---------------------------------------------------------------------------
TERM_NORMALIZATIONS: dict[str, str] = {
    "policyholder": "policy holder",
    "coinsurance": "co insurance",
    "selfinsured": "self insured",
    "sublimit": "sub limit",
    "deductable": "deductible",
    "liabilty": "liability",
    "insurence": "insurance",
    "cliam": "claim",
    "prmeium": "premium",
    "renewel": "renewal",
}

# ---------------------------------------------------------------------------
# Document-type keywords (insurance domain)
# ---------------------------------------------------------------------------
DOC_TYPE_KEYWORDS: dict[str, list[str]] = {
    "policy_wording": ["policy wording", "policy document", "terms and conditions", "coverage terms", "policy terms", "wording"],
    "endorsement": ["endorsement", "amendment", "rider", "addendum", "policy change", "modification"],
    "certificate": ["certificate", "coi", "certificate of insurance", "proof of insurance", "cert"],
    "claim": ["claim", "claim form", "loss notice", "incident report", "claim submission", "fnol"],
    "underwriting": ["underwriting", "risk assessment", "underwriting guide", "risk evaluation", "uw guide"],
    "premium": ["premium", "rate", "pricing", "premium schedule", "rate sheet", "quote"],
    "process": ["process", "procedure", "workflow", "how to", "step by step", "sop", "guide"],
    "announcement": ["announcement", "update", "news", "bulletin", "notice", "memo"],
    "compliance": ["compliance", "regulatory", "audit", "regulation", "requirement", "filing"],
    "renewal": ["renewal", "renew", "expiration", "policy renewal", "renewal notice"],
    "cancellation": ["cancel", "cancellation", "termination", "policy cancellation", "non-renewal"],
    "coverage": ["coverage", "coverage summary", "declarations", "dec page", "limits"],
    "training": ["training", "onboarding", "education", "learning", "course"],
}

# ---------------------------------------------------------------------------
# Category keywords (maps query intent -> preferred chunk categories)
# ---------------------------------------------------------------------------
CATEGORY_KEYWORDS: dict[str, list[str]] = {
    "procedure": ["how to", "steps", "process", "procedure", "workflow", "instructions", "guide"],
    "definition": ["what is", "define", "meaning", "definition", "what does", "what are"],
    "data": ["statistics", "numbers", "data", "metrics", "table", "figures", "amount"],
    "example": ["example", "case study", "illustration", "sample", "instance", "scenario"],
    "reference": ["reference", "citation", "source", "appendix", "annex", "schedule"],
    "narrative": ["background", "history", "overview", "context", "story"],
}

# ---------------------------------------------------------------------------
# Synthesis keywords (triggers LLM synthesis when detected in a query)
# ---------------------------------------------------------------------------
SYNTHESIS_KEYWORDS: list[str] = [
    "summarize",
    "explain",
    "compare",
    "difference",
    "relationship",
    "how does",
    "why",
    "what is the",
    "overview",
    "total",
    "average",
    "trend",
]

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
TABULAR_EXTENSIONS: set[str] = {"csv", "xlsx", "xls"}
SUPPORTED_UPLOAD_EXTENSIONS: set[str] = {"pdf", "docx", "xlsx", "xls", "csv", "pptx"}

## Utilities

In [ ]:
# --- File utilities ---
def get_file_type(file_path: Path) -> str:
    return file_path.suffix.lower().lstrip(".")

def is_tabular(file_type: str) -> bool:
    return file_type.lower() in TABULAR_EXTENSIONS

def generate_document_id(file_path: Path) -> str:
    path_str = str(file_path.absolute())
    return hashlib.sha256(path_str.encode()).hexdigest()[:32]

def compute_file_hash(file_path: Path) -> str:
    sha256 = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            sha256.update(chunk)
    return sha256.hexdigest()

# --- Chunk utilities ---
def build_chunk_context(
    file_name: str,
    doc_type: str,
    heading_path: str,
    entities: dict[str, Any],
    prev_summary: str | None = None,
    next_summary: str | None = None,
    contextual_description: str | None = None,
) -> str:
    lines = [f"Document: {file_name}", f"Type: {doc_type}"]
    if heading_path:
        lines.append(f"Section: {heading_path}")
    if entities:
        entity_strs = [f"{k}={v}" for k, v in list(entities.items())[:5]]
        if entity_strs:
            lines.append(f"Key info: {', '.join(entity_strs)}")
    if settings.enable_chunk_links:
        if prev_summary:
            lines.append(f"Previous context: {prev_summary}")
        if next_summary:
            lines.append(f"Following context: {next_summary}")
    if contextual_description:
        lines.append(f"Context: {contextual_description}")
    return "\n".join(lines)

def generate_chunk_id(document_id: str, chunk_index: int) -> str:
    content = f"{document_id}:{chunk_index}"
    return hashlib.sha256(content.encode()).hexdigest()[:16]

def establish_chunk_links(chunks: list[Chunk]) -> None:
    if not settings.enable_chunk_links or len(chunks) < 2:
        return
    for i, chunk in enumerate(chunks):
        if i > 0:
            chunk.prev_chunk_id = chunks[i - 1].id
        if i < len(chunks) - 1:
            chunk.next_chunk_id = chunks[i + 1].id

def split_text(text: str, chunk_size: int) -> list[str]:
    chunks = []
    current_chunk = ""
    paragraphs = text.split("\n\n")
    for para in paragraphs:
        para = para.strip()
        if not para:
            continue
        if len(current_chunk) + len(para) <= chunk_size:
            current_chunk += para + "\n\n"
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            if len(para) > chunk_size:
                long_para_chunks = _split_long_paragraph(para, chunk_size)
                chunks.extend(long_para_chunks)
                current_chunk = ""
            else:
                current_chunk = para + "\n\n"
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    return chunks

def _split_long_paragraph(text: str, chunk_size: int) -> list[str]:
    chunks = []
    current_chunk = ""
    sentences = text.replace(". ", ".|").replace("? ", "?|").replace("! ", "!|").split("|")
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
        if len(current_chunk) + len(sentence) <= chunk_size:
            current_chunk += sentence + " "
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sentence + " "
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    return chunks

# --- Stopwords ---
STOPWORDS: set[str] = {
    "a", "an", "the", "and", "or", "but", "in", "on", "at", "to", "for",
    "of", "with", "by", "from", "as", "is", "was", "are", "were", "been",
    "be", "have", "has", "had", "do", "does", "did", "will", "would",
    "could", "should", "may", "might", "must", "shall", "can", "need",
    "it", "its", "this", "that", "these", "those", "i", "you", "he",
    "she", "we", "they", "what", "which", "who", "whom", "when", "where",
    "why", "how", "all", "each", "every", "both", "few", "more", "most",
    "other", "some", "such", "no", "nor", "not", "only", "own", "same",
    "so", "than", "too", "very", "just", "also", "now", "here", "there",
    "then", "once", "if", "because", "until", "while", "about", "into",
    "through", "during", "before", "after", "above", "below", "between",
    "under", "again", "further", "any", "being", "having", "doing",
}

## Parsers
### CSV & Excel

In [ ]:
class CSVParser:
    """Parser for CSV files using Pandas."""

    MAX_SAMPLE_ROWS = 10
    MAX_TEXT_ROWS = 100

    def parse(self, file_path: Path) -> ParseResult:
        """Parse a CSV file and extract schema and content."""
        try:
            encoding = self._detect_encoding(file_path)
            df = pd.read_csv(file_path, encoding=encoding)

            column_headers = list(df.columns)
            column_types = self._detect_column_types(df)
            row_count = len(df)

            sample_df = df.head(self.MAX_SAMPLE_ROWS)
            sample_rows = sample_df.values.tolist()

            numeric_stats = self._calculate_stats(df)
            text = self._create_text_representation(df, file_path.name)

            tables = [
                TableData(
                    headers=column_headers,
                    rows=sample_rows,
                    title=f"CSV Data: {file_path.name}",
                )
            ]

            metadata = {
                "file_name": file_path.name,
                "file_type": "csv",
                "encoding": encoding,
                "row_count": row_count,
                "column_count": len(column_headers),
            }

            return ParseResult(
                text=text,
                tables=tables,
                metadata=metadata,
                column_headers=column_headers,
                column_types=column_types,
                sample_rows=sample_rows,
                row_count=row_count,
                numeric_stats=numeric_stats,
                dataframe=df,
            )

        except Exception as e:
            logger.error(f"Failed to parse CSV {file_path}: {e}")
            return ParseResult(
                text=f"[CSV parsing failed: {e}]",
                metadata={
                    "file_name": file_path.name,
                    "file_type": "csv",
                    "error": str(e),
                },
            )

    def _detect_encoding(self, file_path: Path) -> str:
        """Detect file encoding using chardet."""
        try:
            with open(file_path, "rb") as f:
                raw = f.read(10000)
                result = chardet.detect(raw)
                return result.get("encoding", "utf-8") or "utf-8"
        except Exception:
            return "utf-8"

    def _detect_column_types(self, df: pd.DataFrame) -> dict[str, str]:
        """Detect and describe column data types."""
        type_map = {}
        for col in df.columns:
            dtype = df[col].dtype
            if pd.api.types.is_datetime64_any_dtype(dtype):
                type_map[col] = "datetime"
            elif pd.api.types.is_numeric_dtype(dtype):
                if pd.api.types.is_integer_dtype(dtype):
                    type_map[col] = "integer"
                else:
                    type_map[col] = "decimal"
            elif pd.api.types.is_bool_dtype(dtype):
                type_map[col] = "boolean"
            else:
                if self._looks_like_date(df[col]):
                    type_map[col] = "date_string"
                elif self._looks_like_currency(df[col]):
                    type_map[col] = "currency"
                else:
                    type_map[col] = "text"
        return type_map

    def _looks_like_date(self, series: pd.Series) -> bool:
        """Check if a series looks like dates."""
        try:
            sample = series.dropna().head(10)
            if len(sample) == 0:
                return False
            pd.to_datetime(sample)
            return True
        except Exception:
            return False

    def _looks_like_currency(self, series: pd.Series) -> bool:
        """Check if a series looks like currency values."""
        try:
            sample = series.dropna().astype(str).head(10)
            currency_chars = ["$", "\u20ac", "\u00a3", "\u00a5", "\u20b9"]
            return any(
                any(char in val for char in currency_chars) for val in sample
            )
        except Exception:
            return False

    def _calculate_stats(self, df: pd.DataFrame) -> dict[str, dict[str, float]]:
        """Calculate statistics for numeric columns."""
        stats = {}
        numeric_cols = df.select_dtypes(include=["number"]).columns

        for col in numeric_cols:
            try:
                col_stats = df[col].describe()
                stats[col] = {
                    "min": float(col_stats.get("min", 0)),
                    "max": float(col_stats.get("max", 0)),
                    "mean": float(col_stats.get("mean", 0)),
                    "sum": float(df[col].sum()),
                }
            except Exception:
                pass

        return stats

    def _create_text_representation(self, df: pd.DataFrame, file_name: str) -> str:
        """Create a text representation of the CSV for indexing."""
        lines = []

        lines.append(f"CSV File: {file_name}")
        lines.append(f"Total rows: {len(df)}")
        lines.append(f"Columns: {', '.join(df.columns)}")
        lines.append("")

        lines.append("Column Details:")
        for col in df.columns:
            dtype = df[col].dtype
            unique_count = df[col].nunique()
            null_count = df[col].isnull().sum()

            type_desc = self._get_type_description(df[col])
            lines.append(f"  - {col}: {type_desc} ({unique_count} unique values)")

            if unique_count <= 10 and dtype == "object":
                sample_vals = df[col].dropna().unique()[:5].tolist()
                lines.append(f"    Sample values: {', '.join(str(v) for v in sample_vals)}")

        lines.append("")

        numeric_cols = df.select_dtypes(include=["number"]).columns
        if len(numeric_cols) > 0:
            lines.append("Numeric Summaries:")
            for col in numeric_cols:
                try:
                    total = df[col].sum()
                    avg = df[col].mean()
                    lines.append(f"  - {col}: Total={total:,.2f}, Average={avg:,.2f}")
                except Exception:
                    pass
            lines.append("")

        lines.append("Sample Data:")
        for idx, row in df.head(5).iterrows():
            row_parts = []
            for col, val in row.items():
                if pd.notna(val):
                    row_parts.append(f"{col}={val}")
            lines.append(f"  Row {idx + 1}: {', '.join(row_parts)}")

        return "\n".join(lines)

    def _get_type_description(self, series: pd.Series) -> str:
        """Get a human-readable type description."""
        dtype = series.dtype

        if pd.api.types.is_datetime64_any_dtype(dtype):
            return "dates"
        elif pd.api.types.is_numeric_dtype(dtype):
            if pd.api.types.is_integer_dtype(dtype):
                return "integers"
            return "decimal numbers"
        elif pd.api.types.is_bool_dtype(dtype):
            return "true/false values"
        else:
            if self._looks_like_date(series):
                return "date strings"
            elif self._looks_like_currency(series):
                return "currency values"
            return "text"


class ExcelParser:
    """Parser for Excel files (xlsx, xls) using Pandas."""

    MAX_SAMPLE_ROWS = 10

    def parse(self, file_path: Path) -> ParseResult:
        """Parse an Excel file and extract all sheets."""
        try:
            excel_file = pd.ExcelFile(file_path)
            sheet_names = excel_file.sheet_names

            all_tables = []
            all_column_headers = []
            all_column_types = {}
            total_rows = 0
            all_numeric_stats = {}

            text_parts = [f"Excel File: {file_path.name}"]
            text_parts.append(f"Sheets: {', '.join(sheet_names)}")
            text_parts.append("")

            for sheet_name in sheet_names:
                try:
                    df = pd.read_excel(excel_file, sheet_name=sheet_name)

                    if df.empty:
                        text_parts.append(f"Sheet '{sheet_name}': Empty")
                        continue

                    column_headers = list(df.columns)
                    column_types = self._detect_column_types(df)
                    row_count = len(df)
                    total_rows += row_count

                    sample_rows = df.head(self.MAX_SAMPLE_ROWS).values.tolist()

                    all_tables.append(
                        TableData(
                            headers=column_headers,
                            rows=sample_rows,
                            title=f"Sheet: {sheet_name}",
                            sheet_name=sheet_name,
                        )
                    )

                    all_column_headers.extend(column_headers)
                    for col, dtype in column_types.items():
                        all_column_types[f"{sheet_name}.{col}"] = dtype

                    stats = self._calculate_stats(df)
                    for col, col_stats in stats.items():
                        all_numeric_stats[f"{sheet_name}.{col}"] = col_stats

                    text_parts.append(f"--- Sheet: {sheet_name} ---")
                    text_parts.append(f"Rows: {row_count}")
                    text_parts.append(f"Columns: {', '.join(column_headers)}")
                    text_parts.append("")
                    text_parts.append(self._describe_sheet(df, sheet_name))
                    text_parts.append("")

                except Exception as e:
                    logger.warning(f"Failed to parse sheet {sheet_name}: {e}")
                    text_parts.append(f"Sheet '{sheet_name}': Error - {e}")

            metadata = {
                "file_name": file_path.name,
                "file_type": get_file_type(file_path),
                "sheet_count": len(sheet_names),
                "total_rows": total_rows,
            }

            return ParseResult(
                text="\n".join(text_parts),
                tables=all_tables,
                metadata=metadata,
                column_headers=list(set(all_column_headers)),
                column_types=all_column_types,
                row_count=total_rows,
                sheet_names=sheet_names,
                numeric_stats=all_numeric_stats,
            )

        except Exception as e:
            logger.error(f"Failed to parse Excel {file_path}: {e}")
            return ParseResult(
                text=f"[Excel parsing failed: {e}]",
                metadata={
                    "file_name": file_path.name,
                    "file_type": get_file_type(file_path),
                    "error": str(e),
                },
            )

    def _detect_column_types(self, df: pd.DataFrame) -> dict[str, str]:
        """Detect and describe column data types."""
        type_map = {}
        for col in df.columns:
            dtype = df[col].dtype
            if pd.api.types.is_datetime64_any_dtype(dtype):
                type_map[col] = "datetime"
            elif pd.api.types.is_numeric_dtype(dtype):
                if pd.api.types.is_integer_dtype(dtype):
                    type_map[col] = "integer"
                else:
                    type_map[col] = "decimal"
            elif pd.api.types.is_bool_dtype(dtype):
                type_map[col] = "boolean"
            else:
                type_map[col] = "text"
        return type_map

    def _calculate_stats(self, df: pd.DataFrame) -> dict[str, dict[str, float]]:
        """Calculate statistics for numeric columns."""
        stats = {}
        numeric_cols = df.select_dtypes(include=["number"]).columns

        for col in numeric_cols:
            try:
                col_stats = df[col].describe()
                stats[col] = {
                    "min": float(col_stats.get("min", 0)),
                    "max": float(col_stats.get("max", 0)),
                    "mean": float(col_stats.get("mean", 0)),
                    "sum": float(df[col].sum()),
                }
            except Exception:
                pass

        return stats

    def _describe_sheet(self, df: pd.DataFrame, sheet_name: str) -> str:
        """Create a text description of a sheet."""
        lines = []

        lines.append("Column Details:")
        for col in df.columns:
            unique_count = df[col].nunique()
            null_count = df[col].isnull().sum()
            dtype = df[col].dtype

            type_desc = self._get_type_description(df[col])
            lines.append(f"  - {col}: {type_desc} ({unique_count} unique)")

            if unique_count <= 10:
                sample_vals = df[col].dropna().unique()[:5].tolist()
                if sample_vals:
                    lines.append(f"    Values: {', '.join(str(v) for v in sample_vals)}")

        numeric_cols = df.select_dtypes(include=["number"]).columns
        if len(numeric_cols) > 0:
            lines.append("")
            lines.append("Numeric Summaries:")
            for col in numeric_cols:
                try:
                    total = df[col].sum()
                    avg = df[col].mean()
                    min_val = df[col].min()
                    max_val = df[col].max()
                    lines.append(
                        f"  - {col}: Total={total:,.2f}, Avg={avg:,.2f}, "
                        f"Range=[{min_val:,.2f} to {max_val:,.2f}]"
                    )
                except Exception:
                    pass

        lines.append("")
        lines.append("Sample Records:")
        for idx, row in df.head(3).iterrows():
            row_parts = []
            for col, val in row.items():
                if pd.notna(val):
                    row_parts.append(f"{col}={val}")
            if row_parts:
                lines.append(f"  Record: {', '.join(row_parts[:5])}")

        return "\n".join(lines)

    def _get_type_description(self, series: pd.Series) -> str:
        """Get a human-readable type description."""
        dtype = series.dtype

        if pd.api.types.is_datetime64_any_dtype(dtype):
            return "dates"
        elif pd.api.types.is_numeric_dtype(dtype):
            if pd.api.types.is_integer_dtype(dtype):
                return "integers"
            return "decimal numbers"
        elif pd.api.types.is_bool_dtype(dtype):
            return "true/false"
        else:
            return "text"


csv_parser = CSVParser()
excel_parser = ExcelParser()

### DOCX & PPTX

In [ ]:
from docx import Document as DocxDocument
from docx.table import Table as DocxTable


class DocxProcessor:
    """Process DOCX files and extract text as markdown."""

    def extract(self, file_path: Path) -> str:
        """Extract text from a DOCX file and convert to markdown."""
        logger.info(f"Extracting text from DOCX: {file_path}")

        doc = DocxDocument(str(file_path))
        markdown_parts = []
        markdown_parts.append(f"# {file_path.name}\n")

        for element in doc.element.body:
            if element.tag.endswith("p"):
                para = self._find_paragraph(doc, element)
                if para:
                    formatted = self._format_paragraph(para)
                    if formatted:
                        markdown_parts.append(formatted)

            elif element.tag.endswith("tbl"):
                table = self._find_table(doc, element)
                if table:
                    formatted = self._format_table(table)
                    if formatted:
                        markdown_parts.append(formatted)

        markdown = "\n\n".join(markdown_parts)
        logger.info(f"Extracted {len(markdown)} characters from DOCX")
        return markdown

    def _find_paragraph(self, doc, element):
        """Find the paragraph object for an element."""
        for para in doc.paragraphs:
            if para._element is element:
                return para
        return None

    def _find_table(self, doc, element):
        """Find the table object for an element."""
        for table in doc.tables:
            if table._element is element:
                return table
        return None

    def _format_paragraph(self, para) -> str | None:
        """Format a paragraph as markdown."""
        text = para.text.strip()
        if not text:
            return None

        style_name = para.style.name.lower() if para.style else ""

        if "heading 1" in style_name or "title" in style_name:
            return f"# {text}"
        elif "heading 2" in style_name:
            return f"## {text}"
        elif "heading 3" in style_name:
            return f"### {text}"
        elif "heading 4" in style_name:
            return f"#### {text}"
        elif "heading 5" in style_name:
            return f"##### {text}"
        elif "heading 6" in style_name:
            return f"###### {text}"
        elif "list" in style_name or "bullet" in style_name:
            return f"- {text}"
        elif "number" in style_name:
            return f"1. {text}"

        return text

    def _format_table(self, table: DocxTable) -> str | None:
        """Format a table as markdown."""
        if not table.rows:
            return None

        rows = []
        for row in table.rows:
            cells = [cell.text.strip().replace("|", "\\|") for cell in row.cells]
            rows.append("| " + " | ".join(cells) + " |")

        if len(rows) < 1:
            return None

        num_cols = len(table.rows[0].cells)
        separator = "| " + " | ".join(["---"] * num_cols) + " |"

        result = [rows[0], separator] + rows[1:]
        return "\n".join(result)


from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE


class PptxProcessor:
    """Process PPTX files and extract text as markdown."""

    def extract(self, file_path: Path) -> str:
        """Extract text from a PPTX file and convert to markdown."""
        logger.info(f"Extracting text from PPTX: {file_path}")

        prs = Presentation(str(file_path))
        markdown_parts = []
        markdown_parts.append(f"# {file_path.name}\n")

        slide_num = 0
        for slide_num, slide in enumerate(prs.slides, 1):
            slide_parts = []
            slide_parts.append(f"\n---\n## Slide {slide_num}")

            title = self._get_slide_title(slide)
            if title:
                slide_parts.append(f"### {title}")

            for shape in slide.shapes:
                text = self._extract_shape_text(shape)
                if text and text != title:
                    slide_parts.append(text)

            notes = self._get_notes(slide)
            if notes:
                slide_parts.append(f"\n*Speaker Notes:*\n{notes}")

            if len(slide_parts) > 1:
                markdown_parts.append("\n\n".join(slide_parts))

        markdown = "\n\n".join(markdown_parts)
        logger.info(f"Extracted {len(markdown)} characters from PPTX ({slide_num} slides)")
        return markdown

    def _get_slide_title(self, slide) -> str | None:
        """Extract the title from a slide."""
        if slide.shapes.title:
            return slide.shapes.title.text.strip()

        for shape in slide.shapes:
            if shape.has_text_frame and shape.text_frame.paragraphs:
                text = shape.text.strip()
                if text and len(text) < 200:
                    return text

        return None

    def _extract_shape_text(self, shape) -> str | None:
        """Extract text from a shape."""
        if shape.has_text_frame:
            paragraphs = []
            for para in shape.text_frame.paragraphs:
                text = para.text.strip()
                if text:
                    if para.level > 0 or self._is_bullet(para):
                        indent = "  " * para.level
                        paragraphs.append(f"{indent}- {text}")
                    else:
                        paragraphs.append(text)
            if paragraphs:
                return "\n".join(paragraphs)

        if shape.has_table:
            return self._format_table(shape.table)

        if shape.shape_type == MSO_SHAPE_TYPE.GROUP:
            group_texts = []
            for sub_shape in shape.shapes:
                text = self._extract_shape_text(sub_shape)
                if text:
                    group_texts.append(text)
            if group_texts:
                return "\n".join(group_texts)

        return None

    def _is_bullet(self, paragraph) -> bool:
        """Check if a paragraph is a bullet point."""
        try:
            if paragraph.bullet:
                return paragraph.bullet.type is not None
        except AttributeError:
            pass
        return False

    def _format_table(self, table) -> str | None:
        """Format a table as markdown."""
        rows = []
        for row in table.rows:
            cells = [cell.text.strip().replace("|", "\\|") for cell in row.cells]
            rows.append("| " + " | ".join(cells) + " |")

        if len(rows) < 1:
            return None

        num_cols = len(table.rows[0].cells)
        separator = "| " + " | ".join(["---"] * num_cols) + " |"

        result = [rows[0], separator] + rows[1:]
        return "\n".join(result)

    def _get_notes(self, slide) -> str | None:
        """Get speaker notes from a slide."""
        try:
            if slide.has_notes_slide:
                notes_slide = slide.notes_slide
                notes_text = notes_slide.notes_text_frame.text.strip()
                if notes_text:
                    return notes_text
        except AttributeError:
            pass
        return None


docx_processor = DocxProcessor()
pptx_processor = PptxProcessor()

### PDF

In [ ]:
class PageRenderer:
    """Renders PDF pages as images using PyMuPDF."""

    def __init__(self, dpi: int = 150):
        self.dpi = dpi
        self.zoom = dpi / 72  # PDF default is 72 DPI

    def render_pdf_pages(self, pdf_path: Path, output_dir: Path) -> list[Path]:
        """Convert all PDF pages to PNG images."""
        output_dir.mkdir(parents=True, exist_ok=True)
        logger.info(f"Rendering PDF pages: {pdf_path}")

        doc = fitz.open(str(pdf_path))
        page_paths = []

        for i, page in enumerate(doc):
            mat = fitz.Matrix(self.zoom, self.zoom)
            pix = page.get_pixmap(matrix=mat)
            output_path = output_dir / f"page_{i + 1:03d}.png"
            pix.save(str(output_path))
            page_paths.append(output_path)
            logger.debug(f"Rendered page {i + 1} to: {output_path}")

        doc.close()
        logger.info(f"Rendered {len(page_paths)} pages from PDF")
        return page_paths

    def render_single_page(self, pdf_path: Path, page_number: int, output_path: Path) -> Path:
        """Render a single PDF page to an image."""
        doc = fitz.open(str(pdf_path))
        if page_number < 1 or page_number > len(doc):
            doc.close()
            raise ValueError(f"Invalid page number: {page_number}. PDF has {len(doc)} pages.")

        page = doc[page_number - 1]
        mat = fitz.Matrix(self.zoom, self.zoom)
        pix = page.get_pixmap(matrix=mat)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        pix.save(str(output_path))
        doc.close()
        return output_path

    def get_page_count(self, pdf_path: Path) -> int:
        """Get the number of pages in a PDF."""
        doc = fitz.open(str(pdf_path))
        count = len(doc)
        doc.close()
        return count

    def get_page_dimensions(self, pdf_path: Path, page_number: int = 1) -> tuple[int, int]:
        """Get the dimensions of a PDF page at the render DPI."""
        doc = fitz.open(str(pdf_path))
        if page_number < 1 or page_number > len(doc):
            doc.close()
            raise ValueError(f"Invalid page number: {page_number}")
        page = doc[page_number - 1]
        rect = page.rect
        width = int(rect.width * self.zoom)
        height = int(rect.height * self.zoom)
        doc.close()
        return (width, height)


class LayoutDetector:
    """DocLayout-YOLO layout detector stub (graceful import)."""

    def __init__(self):
        self._model = None

    @property
    def model(self):
        """Lazy load the DocLayout-YOLO model."""
        if self._model is None:
            try:
                from doclayout_yolo import YOLOv10
                from huggingface_hub import hf_hub_download

                logger.info("Loading DocLayout-YOLO model...")
                model_path = hf_hub_download(
                    repo_id="juliozhao/DocLayout-YOLO-DocStructBench",
                    filename="doclayout_yolo_docstructbench_imgsz1024.pt",
                )
                self._model = YOLOv10(model_path)
                logger.info("DocLayout-YOLO model loaded successfully")
            except ImportError:
                logger.warning(
                    "doclayout-yolo not installed. Layout detection unavailable. "
                    "Install with: pip install doclayout-yolo huggingface_hub"
                )
                raise
            except Exception as e:
                logger.error(f"Failed to load DocLayout-YOLO model: {e}")
                raise
        return self._model

    def is_available(self) -> bool:
        """Check if layout detection dependencies are installed."""
        try:
            import doclayout_yolo  # noqa: F401
            return True
        except ImportError:
            return False


def extract_pdf_text_simple(file_path: Path) -> str:
    """Fallback PDF text extraction using PyMuPDF get_text('text')."""
    doc = fitz.open(str(file_path))
    pages = []
    for i, page in enumerate(doc):
        text = page.get_text("text")
        if text.strip():
            pages.append(f"--- Page {i + 1} ---\n{text}")
    doc.close()
    return "\n\n".join(pages)


page_renderer = PageRenderer()
layout_detector = LayoutDetector()

## Chunking
### Paragraph & Tabular Chunker

In [ ]:
class Chunker:
    """Creates contextual chunks from parsed documents."""

    def __init__(self, chunk_size: int | None = None, chunk_overlap: int | None = None):
        self.chunk_size = chunk_size or settings.chunk_size
        self.chunk_overlap = chunk_overlap or settings.chunk_overlap

    def chunk_document(
        self,
        parse_result: ParseResult,
        enrichment: EnrichmentResult,
        document_id: str,
        file_name: str,
    ) -> list[Chunk]:
        """Create chunks from a document (PDF, Word, PowerPoint)."""
        chunks = []
        chunk_index = 0

        current_heading_path = ""
        heading_stack: list[tuple[int, str]] = []

        text_chunks = self._split_text(parse_result.text)

        for text in text_chunks:
            if not text.strip():
                continue

            heading_match = self._detect_heading(text, parse_result.headings)
            if heading_match:
                self._update_heading_stack(heading_stack, heading_match)
                current_heading_path = self._build_heading_path(heading_stack)

            context = self._build_context(
                file_name=file_name,
                doc_type=enrichment.document_type,
                heading_path=current_heading_path,
                entities=enrichment.entities,
            )
            contextualized = f"{context}\n\n{text}"

            chunk_id = self._generate_chunk_id(document_id, chunk_index)
            chunks.append(
                Chunk(
                    id=chunk_id,
                    document_id=document_id,
                    text=text,
                    contextualized_text=contextualized,
                    content_type="paragraph",
                    heading_path=current_heading_path if current_heading_path else None,
                    chunk_index=chunk_index,
                )
            )
            chunk_index += 1

        # Add table chunks
        for i, table in enumerate(parse_result.tables):
            table_text = self._table_to_text(table)
            table_desc = (
                enrichment.table_descriptions[i]
                if i < len(enrichment.table_descriptions)
                else f"Table: {table.title or 'Data Table'}"
            )

            context = self._build_context(
                file_name=file_name,
                doc_type=enrichment.document_type,
                heading_path=current_heading_path,
                entities=enrichment.entities,
            )
            contextualized = f"{context}\n\nTable Description: {table_desc}\n\n{table_text}"

            chunk_id = self._generate_chunk_id(document_id, chunk_index)
            chunks.append(
                Chunk(
                    id=chunk_id,
                    document_id=document_id,
                    text=table_text,
                    contextualized_text=contextualized,
                    content_type="table",
                    chunk_index=chunk_index,
                )
            )
            chunk_index += 1

        # Add summary chunk
        if enrichment.summary:
            context = self._build_context(
                file_name=file_name,
                doc_type=enrichment.document_type,
                heading_path="Document Summary",
                entities=enrichment.entities,
            )
            summary_text = f"Summary: {enrichment.summary}"
            contextualized = f"{context}\n\n{summary_text}"

            chunk_id = self._generate_chunk_id(document_id, chunk_index)
            chunks.append(
                Chunk(
                    id=chunk_id,
                    document_id=document_id,
                    text=summary_text,
                    contextualized_text=contextualized,
                    content_type="summary",
                    chunk_index=chunk_index,
                )
            )

        # Establish cross-chunk links
        self._establish_chunk_links(chunks)

        return chunks

    def chunk_tabular(
        self,
        parse_result: ParseResult,
        enrichment: EnrichmentResult,
        document_id: str,
        file_name: str,
    ) -> list[Chunk]:
        """Create chunks from tabular data (CSV, Excel)."""
        chunks = []
        chunk_index = 0

        # File-level summary chunk
        summary_text = enrichment.summary or f"Tabular data file: {file_name}"
        context = self._build_context(
            file_name=file_name,
            doc_type=enrichment.document_type,
            heading_path="File Summary",
            entities=enrichment.entities,
        )
        contextualized = f"{context}\n\n{summary_text}"

        chunk_id = self._generate_chunk_id(document_id, chunk_index)
        chunks.append(
            Chunk(
                id=chunk_id,
                document_id=document_id,
                text=summary_text,
                contextualized_text=contextualized,
                content_type="summary",
                chunk_index=chunk_index,
            )
        )
        chunk_index += 1

        # Schema description chunk
        if parse_result.column_headers:
            schema_parts = [f"Columns in {file_name}:"]
            for col in parse_result.column_headers[:30]:
                dtype = (
                    parse_result.column_types.get(col, "text")
                    if parse_result.column_types
                    else "text"
                )
                schema_parts.append(f"  - {col}: {dtype}")

            schema_text = "\n".join(schema_parts)
            context = self._build_context(
                file_name=file_name,
                doc_type=enrichment.document_type,
                heading_path="Data Schema",
                entities=enrichment.entities,
            )
            contextualized = f"{context}\n\n{schema_text}"

            chunk_id = self._generate_chunk_id(document_id, chunk_index)
            chunks.append(
                Chunk(
                    id=chunk_id,
                    document_id=document_id,
                    text=schema_text,
                    contextualized_text=contextualized,
                    content_type="schema",
                    chunk_index=chunk_index,
                )
            )
            chunk_index += 1

        # Per-sheet chunks for Excel
        for table in parse_result.tables:
            if table.sheet_name:
                sheet_text = f"Sheet: {table.sheet_name}\n"
                sheet_text += f"Columns: {', '.join(table.headers)}\n"
                if table.rows:
                    sheet_text += f"Sample data: {len(table.rows)} rows shown"

                context = self._build_context(
                    file_name=file_name,
                    doc_type=enrichment.document_type,
                    heading_path=f"Sheet: {table.sheet_name}",
                    entities=enrichment.entities,
                )
                contextualized = f"{context}\n\n{sheet_text}"

                chunk_id = self._generate_chunk_id(document_id, chunk_index)
                chunks.append(
                    Chunk(
                        id=chunk_id,
                        document_id=document_id,
                        text=sheet_text,
                        contextualized_text=contextualized,
                        content_type="table",
                        sheet_name=table.sheet_name,
                        chunk_index=chunk_index,
                    )
                )
                chunk_index += 1

        # Add full text representation as chunks
        if parse_result.text:
            text_chunks = self._split_text(parse_result.text)
            for text in text_chunks:
                if not text.strip():
                    continue

                context = self._build_context(
                    file_name=file_name,
                    doc_type=enrichment.document_type,
                    heading_path="Data Content",
                    entities=enrichment.entities,
                )
                contextualized = f"{context}\n\n{text}"

                chunk_id = self._generate_chunk_id(document_id, chunk_index)
                chunks.append(
                    Chunk(
                        id=chunk_id,
                        document_id=document_id,
                        text=text,
                        contextualized_text=contextualized,
                        content_type="paragraph",
                        chunk_index=chunk_index,
                    )
                )
                chunk_index += 1

        # Row-batch chunks for full CSV/Excel data
        if parse_result.dataframe is not None:
            row_batch_chunks = self._create_row_batch_chunks(
                df=parse_result.dataframe,
                column_headers=parse_result.column_headers or [],
                document_id=document_id,
                file_name=file_name,
                enrichment=enrichment,
                start_chunk_index=chunk_index,
            )
            chunks.extend(row_batch_chunks)
            chunk_index += len(row_batch_chunks)

        # Establish cross-chunk links
        self._establish_chunk_links(chunks)

        return chunks

    def _create_row_batch_chunks(
        self,
        df: pd.DataFrame,
        column_headers: list[str],
        document_id: str,
        file_name: str,
        enrichment: EnrichmentResult,
        start_chunk_index: int,
    ) -> list[Chunk]:
        """Create row-batch chunks from a DataFrame."""
        rows_per_chunk = settings.csv_rows_per_chunk
        max_rows = settings.csv_max_rows_to_index
        total_rows = len(df)

        if max_rows > 0 and total_rows > max_rows:
            logger.warning(
                f"CSV has {total_rows} rows, capping at {max_rows} "
                f"(csv_max_rows_to_index={max_rows})"
            )
            df = df.head(max_rows)
            total_rows = max_rows

        header_line = f"Columns: {', '.join(column_headers)}"
        chunks = []
        chunk_index = start_chunk_index

        for batch_start in range(0, total_rows, rows_per_chunk):
            batch_end = min(batch_start + rows_per_chunk, total_rows)
            batch_df = df.iloc[batch_start:batch_end]

            row_lines = []
            for i, (_, row) in enumerate(batch_df.iterrows()):
                row_num = batch_start + i + 1
                parts = []
                for col in column_headers:
                    val = row.get(col, "")
                    if pd.notna(val):
                        parts.append(f"{col}={val}")
                row_lines.append(f"Row {row_num}: {', '.join(parts)}")

            batch_text = f"{header_line}\n" + "\n".join(row_lines)

            if len(batch_text) > self.chunk_size * 2:
                sub_texts = self._split_row_batch_text(header_line, row_lines)
            else:
                sub_texts = [batch_text]

            for text in sub_texts:
                context = self._build_context(
                    file_name=file_name,
                    doc_type=enrichment.document_type,
                    heading_path="Row Data",
                    entities=enrichment.entities,
                )
                contextualized = f"{context}\n\n{text}"

                chunk_id = self._generate_chunk_id(document_id, chunk_index)
                chunks.append(
                    Chunk(
                        id=chunk_id,
                        document_id=document_id,
                        text=text,
                        contextualized_text=contextualized,
                        content_type="row_batch",
                        chunk_index=chunk_index,
                    )
                )
                chunk_index += 1

        logger.info(
            f"Created {len(chunks)} row batch chunks from {total_rows} rows "
            f"for {file_name}"
        )
        return chunks

    def _split_row_batch_text(self, header_line: str, row_lines: list[str]) -> list[str]:
        """Split an oversized row batch into smaller texts on row boundaries."""
        texts = []
        current_lines = [header_line]
        current_len = len(header_line)

        for line in row_lines:
            line_len = len(line) + 1
            if current_len + line_len > self.chunk_size * 2 and len(current_lines) > 1:
                texts.append("\n".join(current_lines))
                current_lines = [header_line]
                current_len = len(header_line)
            current_lines.append(line)
            current_len += line_len

        if len(current_lines) > 1:
            texts.append("\n".join(current_lines))

        return texts

    def _split_text(self, text: str) -> list[str]:
        """Split text into chunks, respecting paragraph and sentence boundaries."""
        return split_text(text, self.chunk_size)

    def _build_context(self, **kwargs) -> str:
        """Build context prefix for a chunk (delegates to build_chunk_context)."""
        return build_chunk_context(**kwargs)

    def _establish_chunk_links(self, chunks: list[Chunk]) -> None:
        """Establish cross-chunk navigation links."""
        establish_chunk_links(chunks)

    def _detect_heading(self, text: str, headings: list[HeadingInfo]) -> HeadingInfo | None:
        """Check if text matches a known heading."""
        text_clean = text.strip().lower()[:100]
        for heading in headings:
            if heading.text.strip().lower() in text_clean:
                return heading
        return None

    def _update_heading_stack(self, stack: list[tuple[int, str]], heading: HeadingInfo) -> None:
        """Update the heading stack with a new heading."""
        while stack and stack[-1][0] >= heading.level:
            stack.pop()
        stack.append((heading.level, heading.text))

    def _build_heading_path(self, stack: list[tuple[int, str]]) -> str:
        """Build heading path from stack."""
        return " > ".join(text for _, text in stack)

    def _table_to_text(self, table) -> str:
        """Convert a table to text representation."""
        lines = []
        if table.title:
            lines.append(f"Table: {table.title}")
        lines.append(f"Headers: {', '.join(table.headers)}")
        for i, row in enumerate(table.rows[:5], 1):
            row_str = ", ".join(str(v)[:50] for v in row)
            lines.append(f"Row {i}: {row_str}")
        if len(table.rows) > 5:
            lines.append(f"... and {len(table.rows) - 5} more rows")
        return "\n".join(lines)

    def _generate_chunk_id(self, document_id: str, chunk_index: int) -> str:
        """Generate a unique chunk ID."""
        return generate_chunk_id(document_id, chunk_index)


def rebuild_contextualized_text(
    chunks: list[Chunk],
    enrichment_results: list[tuple[ChunkMetadata | None, list]],
    file_name: str,
    doc_type: str,
    entities: dict,
) -> list[Chunk]:
    """Rebuild each chunk's contextualized_text using LLM enrichment results.

    Replaces raw text overlap with enrichment-derived context:
    - Previous chunk's LLM-generated summary
    - Next chunk's LLM-generated summary
    - This chunk's contextual_description from enrichment
    """
    for i, chunk in enumerate(chunks):
        prev_summary = None
        next_summary = None
        contextual_desc = None

        if i > 0 and i - 1 < len(enrichment_results):
            prev_meta = enrichment_results[i - 1][0]
            if prev_meta and prev_meta.summary:
                prev_summary = prev_meta.summary

        if i + 1 < len(enrichment_results):
            next_meta = enrichment_results[i + 1][0]
            if next_meta and next_meta.summary:
                next_summary = next_meta.summary

        if i < len(enrichment_results):
            self_meta = enrichment_results[i][0]
            if self_meta and self_meta.contextual_description:
                contextual_desc = self_meta.contextual_description

        context = build_chunk_context(
            file_name=file_name,
            doc_type=doc_type,
            heading_path=chunk.heading_path or "",
            entities=entities,
            prev_summary=prev_summary,
            next_summary=next_summary,
            contextual_description=contextual_desc,
        )

        chunk.contextualized_text = f"{context}\n\n{chunk.text}"

        chunk.prev_chunk_summary = prev_summary
        chunk.next_chunk_summary = next_summary

    return chunks


chunker = Chunker()

### Hierarchical Chunker

In [ ]:
@dataclass
class MarkdownNode:
    """Represents a node in the markdown document tree."""
    level: int  # 0=root, 1=h1, 2=h2, etc.
    heading: str | None = None
    content: str = ""
    children: list["MarkdownNode"] = field(default_factory=list)
    start_line: int = 0
    end_line: int = 0
    node_type: Literal["root", "heading", "paragraph", "table", "list", "figure", "code"] = "paragraph"


@dataclass
class LayoutBox:
    """Represents a layout detection bounding box."""
    label: str  # title, text, table, figure, list, etc.
    bbox: list[float]  # [x1, y1, x2, y2]
    page: int
    text: str | None = None


class HierarchicalChunker:
    """Creates chunks that respect markdown structure and layout detection boxes."""

    def __init__(self, max_chunk_size: int | None = None, min_chunk_size: int = 100):
        self.max_chunk_size = max_chunk_size or settings.max_chunk_size
        self.min_chunk_size = min_chunk_size
        self._chunk_index = 0

    def chunk_document(
        self,
        markdown: str,
        layout_data: list[dict] | None,
        document_id: str,
        file_name: str = "",
        detected_doc_type: str = "unknown",
        entities: dict | None = None,
    ) -> list[Chunk]:
        """Create hierarchical chunks from markdown and optional layout data."""
        self._chunk_index = 0
        entities = entities or {}

        root = self._parse_markdown_tree(markdown)
        layout_boxes = self._parse_layout_data(layout_data) if layout_data else []

        chunks = self._tree_to_chunks(
            root,
            document_id=document_id,
            file_name=file_name,
            detected_doc_type=detected_doc_type,
            entities=entities,
            layout_boxes=layout_boxes,
        )

        establish_chunk_links(chunks)
        return chunks

    def _parse_markdown_tree(self, markdown: str) -> MarkdownNode:
        """Parse markdown into a tree structure based on headings."""
        root = MarkdownNode(level=0, node_type="root")
        lines = markdown.split("\n")

        stack: list[MarkdownNode] = [root]
        current_content: list[str] = []
        content_start_line = 0
        in_code_block = False
        in_table = False

        for i, line in enumerate(lines):
            if line.strip().startswith("```"):
                in_code_block = not in_code_block
                current_content.append(line)
                continue

            if in_code_block:
                current_content.append(line)
                continue

            if line.strip().startswith("|") and not in_table:
                if current_content:
                    self._flush_content(stack[-1], current_content, content_start_line, i - 1)
                    current_content = []
                in_table = True
                content_start_line = i

            if in_table:
                if not line.strip().startswith("|") and line.strip():
                    self._flush_content(stack[-1], current_content, content_start_line, i - 1, "table")
                    current_content = []
                    in_table = False
                    content_start_line = i
                else:
                    current_content.append(line)
                    continue

            heading_match = re.match(r"^(#{1,6})\s+(.+)$", line)
            if heading_match:
                if current_content:
                    self._flush_content(stack[-1], current_content, content_start_line, i - 1)
                    current_content = []

                level = len(heading_match.group(1))
                heading_text = heading_match.group(2).strip()

                new_node = MarkdownNode(
                    level=level,
                    heading=heading_text,
                    node_type="heading",
                    start_line=i,
                )

                while len(stack) > 1 and stack[-1].level >= level:
                    stack.pop()

                stack[-1].children.append(new_node)
                stack.append(new_node)
                content_start_line = i + 1
            else:
                if re.match(r"^\s*[-*+]\s+", line) or re.match(r"^\s*\d+\.\s+", line):
                    if not current_content or not self._is_list_content("\n".join(current_content)):
                        if current_content:
                            self._flush_content(stack[-1], current_content, content_start_line, i - 1)
                            current_content = []
                            content_start_line = i

                current_content.append(line)

        if current_content:
            node_type = "table" if in_table else ("list" if self._is_list_content("\n".join(current_content)) else "paragraph")
            self._flush_content(stack[-1], current_content, content_start_line, len(lines) - 1, node_type)

        return root

    def _is_list_content(self, content: str) -> bool:
        """Check if content is primarily list items."""
        lines = [l for l in content.strip().split("\n") if l.strip()]
        if not lines:
            return False
        list_lines = sum(1 for l in lines if re.match(r"^\s*[-*+]\s+", l) or re.match(r"^\s*\d+\.\s+", l))
        return list_lines > len(lines) / 2

    def _flush_content(
        self,
        parent: MarkdownNode,
        content_lines: list[str],
        start_line: int,
        end_line: int,
        node_type: str = "paragraph",
    ) -> None:
        """Add accumulated content as a child node."""
        content = "\n".join(content_lines).strip()
        if not content:
            return

        if "![" in content or "<img" in content.lower():
            node_type = "figure"
        if content.startswith("```"):
            node_type = "code"

        child = MarkdownNode(
            level=parent.level + 1,
            content=content,
            node_type=node_type,
            start_line=start_line,
            end_line=end_line,
        )
        parent.children.append(child)

    def _parse_layout_data(self, layout_data: list[dict]) -> list[LayoutBox]:
        """Parse layout detection data into LayoutBox objects."""
        boxes = []
        for item in layout_data:
            if isinstance(item, dict):
                if "regions" in item:
                    page_num = item.get("page", 0)
                    for region in item.get("regions", []):
                        boxes.append(LayoutBox(
                            label=region.get("label", "text"),
                            bbox=region.get("bbox", [0, 0, 0, 0]),
                            page=page_num,
                            text=region.get("text"),
                        ))
                else:
                    boxes.append(LayoutBox(
                        label=item.get("label", "text"),
                        bbox=item.get("bbox", [0, 0, 0, 0]),
                        page=item.get("page", 0),
                        text=item.get("text"),
                    ))
        return boxes

    def _tree_to_chunks(
        self,
        node: MarkdownNode,
        document_id: str,
        file_name: str,
        detected_doc_type: str,
        entities: dict,
        layout_boxes: list[LayoutBox],
        parent_chunk_id: str | None = None,
        heading_path: list[str] | None = None,
    ) -> list[Chunk]:
        """Convert markdown tree to chunks with hierarchy."""
        chunks = []
        heading_path = heading_path or []

        if node.heading:
            new_heading_path = heading_path + [node.heading]

            if node.children or node.content:
                chunk = self._create_chunk(
                    document_id=document_id,
                    text=node.heading,
                    content_type=self._get_content_type(node.level, node.node_type),
                    hierarchy_level=node.level,
                    parent_chunk_id=parent_chunk_id,
                    heading_path=" > ".join(new_heading_path),
                    file_name=file_name,
                    detected_doc_type=detected_doc_type,
                    entities=entities,
                    layout_label=self._find_layout_label(node, layout_boxes),
                )
                chunks.append(chunk)
                parent_chunk_id = chunk.id
        else:
            new_heading_path = heading_path

        if node.content:
            content_chunks = self._split_content(node.content, node.level, node.node_type)

            for content in content_chunks:
                chunk = self._create_chunk(
                    document_id=document_id,
                    text=content,
                    content_type=self._get_content_type(node.level, node.node_type),
                    hierarchy_level=node.level,
                    parent_chunk_id=parent_chunk_id,
                    heading_path=" > ".join(new_heading_path) if new_heading_path else None,
                    file_name=file_name,
                    detected_doc_type=detected_doc_type,
                    entities=entities,
                    layout_label=self._find_layout_label(node, layout_boxes),
                )
                chunks.append(chunk)

        for child in node.children:
            child_chunks = self._tree_to_chunks(
                child,
                document_id=document_id,
                file_name=file_name,
                detected_doc_type=detected_doc_type,
                entities=entities,
                layout_boxes=layout_boxes,
                parent_chunk_id=parent_chunk_id,
                heading_path=new_heading_path,
            )
            chunks.extend(child_chunks)

        return chunks

    def _split_content(self, content: str, level: int, node_type: str) -> list[str]:
        """Split content into chunks if it exceeds max size."""
        if len(content) <= self.max_chunk_size:
            return [content]

        chunks = []
        paragraphs = re.split(r"\n\n+", content)

        current_chunk = ""
        for para in paragraphs:
            if len(current_chunk) + len(para) + 2 <= self.max_chunk_size:
                current_chunk += ("\n\n" if current_chunk else "") + para
            else:
                if current_chunk:
                    chunks.append(current_chunk)

                if len(para) > self.max_chunk_size:
                    sentences = re.split(r"(?<=[.!?])\s+", para)
                    current_chunk = ""
                    for sentence in sentences:
                        if len(current_chunk) + len(sentence) + 1 <= self.max_chunk_size:
                            current_chunk += (" " if current_chunk else "") + sentence
                        else:
                            if current_chunk:
                                chunks.append(current_chunk)
                            current_chunk = sentence
                else:
                    current_chunk = para

        if current_chunk:
            chunks.append(current_chunk)

        return chunks

    def _get_content_type(self, level: int, node_type: str) -> str:
        """Map markdown node type to chunk content type."""
        if node_type == "heading":
            if level == 1:
                return "title"
            return "section_header"
        if node_type == "table":
            return "table"
        if node_type == "list":
            return "list"
        if node_type == "figure":
            return "figure"
        return "paragraph"

    def _find_layout_label(self, node: MarkdownNode, layout_boxes: list[LayoutBox]) -> str | None:
        """Find matching layout box label for a markdown node."""
        if not layout_boxes:
            return None

        node_text = (node.heading or node.content or "").lower()[:100]
        if not node_text:
            return None

        for box in layout_boxes:
            if box.text and node_text in box.text.lower():
                return box.label

        return None

    def _create_chunk(
        self,
        document_id: str,
        text: str,
        content_type: str,
        hierarchy_level: int,
        parent_chunk_id: str | None,
        heading_path: str | None,
        file_name: str,
        detected_doc_type: str,
        entities: dict,
        layout_label: str | None = None,
        bbox: list[float] | None = None,
        page: int | None = None,
    ) -> Chunk:
        """Create a chunk with contextualized text."""
        chunk_id = generate_chunk_id(document_id, self._chunk_index)
        self._chunk_index += 1

        context = build_chunk_context(
            file_name=file_name,
            doc_type=detected_doc_type,
            heading_path=heading_path or "",
            entities=entities,
        )
        contextualized_text = f"{context}\n\n{text}" if context else text

        return Chunk(
            id=chunk_id,
            document_id=document_id,
            text=text,
            contextualized_text=contextualized_text,
            content_type=content_type,
            chunk_index=self._chunk_index - 1,
            parent_chunk_id=parent_chunk_id,
            hierarchy_level=hierarchy_level,
            heading_path=heading_path,
            layout_label=layout_label,
            bbox=bbox,
            page=page,
            entities=entities,
        )


hierarchical_chunker = HierarchicalChunker()

### Semantic Chunker

In [ ]:
class SemanticChunker:
    """Detects semantic boundaries and refines chunks based on topic similarity."""

    def __init__(self, embedder, similarity_threshold: float | None = None):
        self.embedder = embedder
        self.similarity_threshold = similarity_threshold or settings.semantic_similarity_threshold

    def detect_breakpoints(self, text: str) -> list[int]:
        """Detect semantic breakpoints in text using embedding similarity."""
        sentences = self._split_sentences(text)
        if len(sentences) < 3:
            return []

        sentence_texts = [s["text"] for s in sentences]
        embeddings = self.embedder.embed_batch(sentence_texts)

        similarities = []
        for i in range(len(embeddings) - 1):
            sim = self.embedder.similarity(embeddings[i], embeddings[i + 1])
            similarities.append(sim)

        breakpoints = []
        for i, sim in enumerate(similarities):
            if sim < self.similarity_threshold:
                breakpoints.append(sentences[i + 1]["start"])

        return breakpoints

    def refine_chunks(
        self,
        hierarchical_chunks: list[Chunk],
        max_chunk_size: int | None = None,
    ) -> list[Chunk]:
        """Refine chunks by splitting large ones at semantic boundaries."""
        max_size = max_chunk_size or settings.max_chunk_size
        refined_chunks = []

        for chunk in hierarchical_chunks:
            if len(chunk.text) <= max_size:
                refined_chunks.append(chunk)
                continue

            breakpoints = self.detect_breakpoints(chunk.text)

            if not breakpoints:
                refined_chunks.append(chunk)
                continue

            splits = self._split_at_breakpoints(chunk, breakpoints, max_size)
            refined_chunks.extend(splits)

        self._add_semantic_similarity(refined_chunks)
        return refined_chunks

    def _split_sentences(self, text: str) -> list[dict]:
        """Split text into sentences with position tracking."""
        sentence_pattern = r'(?<=[.!?])\s+(?=[A-Z])|(?<=[.!?])\s*\n+'

        sentences = []
        last_end = 0

        for match in re.finditer(sentence_pattern, text):
            sentence_text = text[last_end:match.start() + 1].strip()
            if sentence_text and len(sentence_text) > 10:
                sentences.append({
                    "text": sentence_text,
                    "start": last_end,
                    "end": match.start() + 1,
                })
            last_end = match.end()

        if last_end < len(text):
            final = text[last_end:].strip()
            if final and len(final) > 10:
                sentences.append({
                    "text": final,
                    "start": last_end,
                    "end": len(text),
                })

        return sentences

    def _split_at_breakpoints(
        self,
        chunk: Chunk,
        breakpoints: list[int],
        max_size: int,
    ) -> list[Chunk]:
        """Split a chunk at the given breakpoints."""
        splits = []
        text = chunk.text
        last_pos = 0
        chunk_index = chunk.chunk_index

        for bp in breakpoints:
            segment = text[last_pos:bp].strip()
            if segment:
                if len(segment) >= 50:
                    new_chunk = Chunk(
                        id=f"{chunk.id}_{len(splits)}",
                        document_id=chunk.document_id,
                        text=segment,
                        contextualized_text=self._rebuild_context(chunk, segment),
                        content_type=chunk.content_type,
                        page=chunk.page,
                        sheet_name=chunk.sheet_name,
                        heading_path=chunk.heading_path,
                        entities=chunk.entities,
                        chunk_index=chunk_index,
                        parent_chunk_id=chunk.parent_chunk_id,
                        hierarchy_level=chunk.hierarchy_level,
                        bbox=chunk.bbox,
                        layout_label=chunk.layout_label,
                        is_semantic_boundary=len(splits) > 0,
                    )
                    splits.append(new_chunk)
                    chunk_index += 1

            last_pos = bp

        remaining = text[last_pos:].strip()
        if remaining and len(remaining) >= 50:
            new_chunk = Chunk(
                id=f"{chunk.id}_{len(splits)}",
                document_id=chunk.document_id,
                text=remaining,
                contextualized_text=self._rebuild_context(chunk, remaining),
                content_type=chunk.content_type,
                page=chunk.page,
                sheet_name=chunk.sheet_name,
                heading_path=chunk.heading_path,
                entities=chunk.entities,
                chunk_index=chunk_index,
                parent_chunk_id=chunk.parent_chunk_id,
                hierarchy_level=chunk.hierarchy_level,
                bbox=chunk.bbox,
                layout_label=chunk.layout_label,
                is_semantic_boundary=len(splits) > 0,
            )
            splits.append(new_chunk)

        if not splits:
            return [chunk]

        return splits

    def _rebuild_context(self, original_chunk: Chunk, new_text: str) -> str:
        """Rebuild contextualized text for a split chunk."""
        original_ctx = original_chunk.contextualized_text
        original_text = original_chunk.text

        if original_text in original_ctx:
            text_start = original_ctx.find(original_text)
            context_prefix = original_ctx[:text_start] if text_start > 0 else ""
            return f"{context_prefix}{new_text}" if context_prefix else new_text

        return new_text

    def _add_semantic_similarity(self, chunks: list[Chunk]) -> None:
        """Add semantic similarity scores between consecutive chunks."""
        if len(chunks) < 2:
            return

        doc_chunks: dict[str, list[Chunk]] = {}
        for chunk in chunks:
            if chunk.document_id not in doc_chunks:
                doc_chunks[chunk.document_id] = []
            doc_chunks[chunk.document_id].append(chunk)

        for doc_id, doc_chunk_list in doc_chunks.items():
            if len(doc_chunk_list) < 2:
                continue

            sorted_chunks = sorted(doc_chunk_list, key=lambda c: c.chunk_index)

            texts = [c.text for c in sorted_chunks]
            embeddings = self.embedder.embed_batch(texts)

            for i in range(1, len(sorted_chunks)):
                sim = self.embedder.similarity(embeddings[i - 1], embeddings[i])
                sorted_chunks[i].semantic_similarity_prev = sim

                if sim < self.similarity_threshold:
                    sorted_chunks[i].is_semantic_boundary = True


def create_semantic_chunker(embedder) -> SemanticChunker:
    """Factory function to create a semantic chunker."""
    return SemanticChunker(embedder)

## LLM Client

In [ ]:
# ---------------------------------------------------------------------------
# LLM Client — abstract base + OpenAI / Gemini implementations
# ---------------------------------------------------------------------------

# Pricing per 1M tokens: (input, output)
OPENAI_PRICING: dict[str, tuple[float, float]] = {
    "gpt-5.2": (1.75, 14.00), "gpt-5.1": (1.25, 10.00), "gpt-5": (1.25, 10.00),
    "gpt-5-mini": (0.25, 2.00), "gpt-5-nano": (0.05, 0.40),
    "gpt-4.1": (2.00, 8.00), "gpt-4.1-mini": (0.40, 1.60), "gpt-4.1-nano": (0.10, 0.40),
    "gpt-4o": (2.50, 10.00), "gpt-4o-mini": (0.15, 0.60),
    "o3": (2.00, 8.00), "o4-mini": (1.10, 4.40),
}

GEMINI_PRICING: dict[str, tuple[float, float]] = {
    "gemini-1.5-flash": (0.075, 0.30), "gemini-1.5-pro": (1.25, 5.00),
    "gemini-2.0-flash": (0.10, 0.40), "gemini-2.5-flash": (0.15, 0.60),
    "gemini-2.5-pro": (1.25, 10.00),
}


def _get_pricing(model: str, pricing_table: dict[str, tuple[float, float]]) -> tuple[float, float]:
    if model in pricing_table:
        return pricing_table[model]
    for key in sorted(pricing_table, key=len, reverse=True):
        if model.startswith(key):
            return pricing_table[key]
    return (0.0, 0.0)


class LLMClient(ABC):
    """Abstract base class for LLM clients."""

    @abstractmethod
    async def complete(self, prompt: str, system_prompt: str | None = None,
                       max_tokens: int = 2000, temperature: float = 0.3) -> str:
        pass

    async def complete_json(self, prompt: str, system_prompt: str | None = None,
                            max_tokens: int = 2000) -> dict[str, Any]:
        response = await self.complete(prompt=prompt, system_prompt=system_prompt,
                                        max_tokens=max_tokens, temperature=0.2)
        try:
            if "```json" in response:
                start = response.find("```json") + 7
                end = response.find("```", start)
                if end > start:
                    response = response[start:end].strip()
            elif "```" in response:
                start = response.find("```") + 3
                end = response.find("```", start)
                if end > start:
                    response = response[start:end].strip()
            return json.loads(response)
        except json.JSONDecodeError as e:
            logger.warning(f"Failed to parse JSON response: {e}")
            return {}


class OpenAIClient(LLMClient):
    def __init__(self, api_key: str | None = None, model: str | None = None):
        self.api_key = api_key or settings.openai_api_key
        self.model = model or settings.openai_model
        self._client: AsyncOpenAI | None = None

    @property
    def client(self) -> AsyncOpenAI:
        if self._client is None:
            self._client = AsyncOpenAI(api_key=self.api_key)
        return self._client

    async def complete(self, prompt: str, system_prompt: str | None = None,
                       max_tokens: int = 2000, temperature: float = 0.3) -> str:
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": prompt})
        try:
            response = await self.client.chat.completions.create(
                model=self.model, messages=messages,
                max_completion_tokens=max_tokens, temperature=temperature,
            )
            if response.usage:
                u = response.usage
                price_in, price_out = _get_pricing(self.model, OPENAI_PRICING)
                cost = (u.prompt_tokens / 1e6) * price_in + (u.completion_tokens / 1e6) * price_out
                logger.info(f"[LLM] model={self.model} tokens={u.total_tokens} cost=${cost:.6f}")
            return response.choices[0].message.content or ""
        except Exception as e:
            logger.error(f"OpenAI API error: {e}")
            raise


class GeminiClient(LLMClient):
    def __init__(self, api_key: str | None = None, model: str | None = None):
        self.api_key = api_key or settings.gemini_api_key
        self.model = model or settings.gemini_model
        self._model_instance = None

    def _get_model(self):
        if self._model_instance is None:
            import google.generativeai as genai
            genai.configure(api_key=self.api_key)
            self._model_instance = genai.GenerativeModel(self.model)
        return self._model_instance

    async def complete(self, prompt: str, system_prompt: str | None = None,
                       max_tokens: int = 2000, temperature: float = 0.3) -> str:
        model = self._get_model()
        full_prompt = f"{system_prompt}\n\n{prompt}" if system_prompt else prompt
        try:
            response = await model.generate_content_async(
                full_prompt, generation_config={"max_output_tokens": max_tokens, "temperature": temperature})
            if hasattr(response, "usage_metadata") and response.usage_metadata:
                um = response.usage_metadata
                pt = getattr(um, "prompt_token_count", 0) or 0
                ct = getattr(um, "candidates_token_count", 0) or 0
                price_in, price_out = _get_pricing(self.model, GEMINI_PRICING)
                cost = (pt / 1e6) * price_in + (ct / 1e6) * price_out
                logger.info(f"[LLM] model={self.model} tokens={pt+ct} cost=${cost:.6f}")
            return response.text or ""
        except Exception as e:
            logger.error(f"Gemini API error: {e}")
            raise


def get_llm_client(purpose: str = "default") -> LLMClient:
    model_overrides = {"enrichment": settings.enrichment_model or None,
                       "synthesis": settings.synthesis_model or None}
    model = model_overrides.get(purpose)
    if settings.llm_provider == "openai":
        return OpenAIClient(model=model)
    elif settings.llm_provider == "gemini":
        return GeminiClient(model=model)
    raise ValueError(f"Unknown LLM provider: {settings.llm_provider}")

def get_enrichment_llm_client() -> LLMClient:
    return get_llm_client("enrichment")

def get_synthesis_llm_client() -> LLMClient:
    return get_llm_client("synthesis")


## Prompts & Document Enrichment

In [ ]:
# ---------------------------------------------------------------------------
# LLM Prompt Templates
# ---------------------------------------------------------------------------

DOCUMENT_ENRICHMENT_SYSTEM_PROMPT = """You are an expert document analyst. Your task is to analyze documents and extract key information in a structured format. Always respond with valid JSON only, no explanations."""

DOCUMENT_ENRICHMENT_PROMPT = """Analyze this document and return a JSON object with the following structure:

```json
{{
  "document_type": "string - policy_wording, endorsement, certificate_of_insurance, claim_form, underwriting_guide, premium_schedule, coverage_summary, process_document, announcement, training_material, compliance_document, regulatory_filing, or other",
  "summary": "string - A 2-3 sentence summary",
  "entities": {{ "key": "value pairs of ALL important information" }},
  "key_topics": ["list", "of", "main", "topics"],
  "table_descriptions": ["Description of any tables"]
}}
```

DOCUMENT INFORMATION:
- File name: {file_name}
- File type: {file_type}

DOCUMENT CONTENT:
{content}

Respond with ONLY the JSON object."""

TABULAR_ENRICHMENT_PROMPT = """Analyze this tabular data file and return a JSON object:

```json
{{
  "document_type": "string - premium_data, claims_data, policy_list, loss_run_report, or other",
  "summary": "string - A 2-3 sentence summary",
  "entities": {{ "data_columns": [], "date_range": "", "total_records": "", "key_metrics": "", "categories": "" }},
  "key_topics": ["list"],
  "table_descriptions": ["Description"]
}}
```

FILE INFORMATION:
- File name: {file_name}
- File type: {file_type}
- Row count: {row_count}
- Columns: {columns}

COLUMN TYPES:
{column_types}

SAMPLE DATA:
{sample_data}

NUMERIC STATISTICS:
{numeric_stats}

Respond with ONLY the JSON object."""

QUERY_UNDERSTANDING_PROMPT = """Analyze this search query and extract the user's intent:

Query: {query}

Return a JSON object:
```json
{{
  "search_terms": "string",
  "doc_type_filter": "string or null",
  "entity_filters": {{ }},
  "needs_synthesis": true,
  "query_type": "factual | exploratory | comparative | aggregation"
}}
```

Respond with ONLY the JSON object."""

RESPONSE_SYNTHESIS_PROMPT = """Answer the user's question using ONLY the search results below. Do not invent information.

QUESTION: {query}

SEARCH RESULTS:
{results}

Rules:
1. Answer ONLY what was asked. Prefer higher-scoring results.
2. If no result directly answers, reply: "No exact match was found."
3. Citations: cite clause/section from within the content. Every claim needs an inline citation.
4. Note temporal context.

Format (Markdown): Direct 1-2 sentence answer. Use ### headings for sub-topics, **bold** for key terms, bullets for lists. Be concise."""


# ---------------------------------------------------------------------------
# Entity Extractor (Document-level enrichment)
# ---------------------------------------------------------------------------

class EntityExtractor:
    MAX_CONTENT_LENGTH = 6000

    def __init__(self, llm_client: LLMClient | None = None):
        self._llm_client = llm_client

    @property
    def llm_client(self) -> LLMClient:
        if self._llm_client is None:
            self._llm_client = get_llm_client()
        return self._llm_client

    async def enrich(self, parse_result: ParseResult, file_name: str, file_type: str) -> EnrichmentResult:
        try:
            if is_tabular(file_type):
                return await self._enrich_tabular(parse_result, file_name, file_type)
            return await self._enrich_document(parse_result, file_name, file_type)
        except Exception as e:
            logger.error(f"Enrichment failed for {file_name}: {e}")
            return EnrichmentResult(document_type="unknown", summary=f"Document: {file_name}",
                                    success=False, error=str(e))

    async def _enrich_document(self, pr, file_name, file_type):
        content = pr.text[:self.MAX_CONTENT_LENGTH]
        if len(pr.text) > self.MAX_CONTENT_LENGTH:
            content += "\n[... truncated ...]"
        if pr.tables:
            content += "\n\nTABLES:"
            for i, t in enumerate(pr.tables[:3], 1):
                content += f"\nTable {i}: {t.title or 'Untitled'} | Headers: {', '.join(t.headers)}"
        prompt = DOCUMENT_ENRICHMENT_PROMPT.format(file_name=file_name, file_type=file_type, content=content)
        result = await self.llm_client.complete_json(prompt=prompt, system_prompt=DOCUMENT_ENRICHMENT_SYSTEM_PROMPT)
        return self._parse(result)

    async def _enrich_tabular(self, pr, file_name, file_type):
        col_types = ""
        if pr.column_types:
            for c, d in list(pr.column_types.items())[:20]:
                col_types += f"  - {c}: {d}\n"
        sample = ""
        if pr.sample_rows and pr.column_headers:
            sample += f"Headers: {', '.join(pr.column_headers[:10])}\n"
            for i, row in enumerate(pr.sample_rows[:5], 1):
                sample += f"Row {i}: {', '.join(str(v)[:50] for v in row[:10])}\n"
        stats = ""
        if pr.numeric_stats:
            for c, s in list(pr.numeric_stats.items())[:10]:
                stats += f"  - {c}: sum={s.get('sum',0):,.2f}, mean={s.get('mean',0):,.2f}\n"
        prompt = TABULAR_ENRICHMENT_PROMPT.format(
            file_name=file_name, file_type=file_type, row_count=pr.row_count or "Unknown",
            columns=", ".join(pr.column_headers or [])[:500],
            column_types=col_types or "N/A", sample_data=sample or "N/A",
            numeric_stats=stats or "N/A")
        result = await self.llm_client.complete_json(prompt=prompt, system_prompt=DOCUMENT_ENRICHMENT_SYSTEM_PROMPT)
        return self._parse(result)

    def _parse(self, result):
        if not result:
            return EnrichmentResult(success=False, error="Empty LLM response")
        return EnrichmentResult(
            document_type=result.get("document_type", "unknown"), summary=result.get("summary", ""),
            entities=result.get("entities", {}), key_topics=result.get("key_topics", []),
            table_descriptions=result.get("table_descriptions", []), success=True)


entity_extractor = EntityExtractor()


## Chunk Enrichment

In [ ]:
# ---------------------------------------------------------------------------
# Chunk Enrichment — LLM-based metadata extraction for chunks
# ---------------------------------------------------------------------------

ENRICHMENT_SYSTEM_PROMPT = """You are an expert document analyzer. Extract structured metadata from text chunks. Return valid JSON only."""

ENRICHMENT_PROMPT_TEMPLATE = """Analyze this text chunk and return a JSON object:

Document context:
- File: {file_name}
- Type: {doc_type}
- Section: {heading_path}
- Document summary: {doc_summary}
- Key entities: {doc_entities}

Text chunk:
"""
{chunk_text}
"""

Return JSON with:
{{
  "title": "3-8 word title",
  "summary": "1-2 sentence summary",
  "keywords": ["keyword1", ...],
  "entities": {{
    "policy_numbers": [], "claim_numbers": [], "insured_names": [],
    "coverage_types": [], "premium_amounts": [], "deductibles": [],
    "coverage_limits": [], "effective_dates": [], "expiration_dates": [],
    "agents": [], "carriers": [], "risk_types": [], "locations": [],
    "exclusions": [], "conditions": []
  }},
  "category": "definition|procedure|data|narrative|example|reference",
  "questions": ["5 specific hypothetical questions this chunk answers"],
  "contextual_description": "2-3 sentences on chunk's role in document",
  "temporal_context": "Temporal info or null"
}}"""


class ChunkEnricher:
    def __init__(self, llm_client: LLMClient | None = None, batch_size: int | None = None,
                 questions_per_chunk: int | None = None):
        self._llm_client = llm_client
        self.batch_size = batch_size or settings.enrichment_batch_size
        self.questions_per_chunk = questions_per_chunk or settings.questions_per_chunk

    @property
    def llm_client(self) -> LLMClient:
        if self._llm_client is None:
            self._llm_client = get_enrichment_llm_client()
        return self._llm_client

    async def enrich_chunk(self, chunk: Chunk, doc_context: dict | None = None):
        doc_context = doc_context or {}
        prompt = ENRICHMENT_PROMPT_TEMPLATE.format(
            file_name=doc_context.get("file_name", "Unknown"),
            doc_type=doc_context.get("detected_doc_type", "unknown"),
            heading_path=chunk.heading_path or "N/A",
            chunk_text=chunk.text[:3000],
            doc_summary=doc_context.get("summary", "N/A"),
            doc_entities=self._format_doc_entities(doc_context.get("entities", {})))
        try:
            result = await self.llm_client.complete_json(prompt=prompt, system_prompt=ENRICHMENT_SYSTEM_PROMPT, max_tokens=1500)
            return self._parse_metadata(chunk.id, result), self._parse_questions(chunk.id, result)
        except Exception as e:
            logger.error(f"Failed to enrich chunk {chunk.id}: {e}")
            return self._empty_metadata(chunk.id), []

    async def enrich_batch(self, chunks: list[Chunk], doc_context: dict | None = None,
                           progress_callback=None):
        results = []
        total = len(chunks)
        for i in range(0, total, self.batch_size):
            batch = chunks[i:i + self.batch_size]
            batch_results = await asyncio.gather(
                *[self.enrich_chunk(c, doc_context) for c in batch], return_exceptions=True)
            for j, result in enumerate(batch_results):
                if isinstance(result, Exception):
                    results.append((self._empty_metadata(batch[j].id), []))
                else:
                    results.append(result)
            if progress_callback:
                progress_callback(min(i + self.batch_size, total), total)
            if i + self.batch_size < total:
                await asyncio.sleep(0.5)
        return results

    def _parse_metadata(self, chunk_id, result):
        return ChunkMetadata(
            chunk_id=chunk_id, title=result.get("title"), summary=result.get("summary"),
            keywords=result.get("keywords", [])[:10],
            entities=self._normalize_entities(result.get("entities", {})),
            category=self._validate_category(result.get("category")),
            contextual_description=result.get("contextual_description"),
            temporal_context=result.get("temporal_context"), enriched_at=datetime.utcnow())

    def _parse_questions(self, chunk_id, result):
        return [ChunkQuestion(chunk_id=chunk_id, question=q.strip())
                for q in result.get("questions", [])[:self.questions_per_chunk]
                if isinstance(q, str) and q.strip()]

    def _normalize_entities(self, entities):
        if not entities:
            return {}
        valid_keys = ["policy_numbers", "claim_numbers", "insured_names", "coverage_types",
                      "premium_amounts", "deductibles", "coverage_limits", "effective_dates",
                      "expiration_dates", "agents", "carriers", "risk_types", "locations",
                      "exclusions", "conditions"]
        return {k: list(set(str(v).strip() for v in entities[k] if v))
                for k in valid_keys if k in entities and isinstance(entities[k], list)}

    def _validate_category(self, cat):
        valid = {"definition", "procedure", "data", "narrative", "example", "reference"}
        return cat.lower() if cat and cat.lower() in valid else None

    def _format_doc_entities(self, entities):
        if not entities:
            return "N/A"
        parts = []
        for k, v in list(entities.items())[:8]:
            if isinstance(v, list):
                v = ", ".join(str(x) for x in v[:3])
            parts.append(f"{k}: {v}")
        return "; ".join(parts) if parts else "N/A"

    def _empty_metadata(self, chunk_id):
        return ChunkMetadata(chunk_id=chunk_id, keywords=[], entities={}, enriched_at=datetime.utcnow())


chunk_enricher = ChunkEnricher()


## Embedder

In [ ]:
# ---------------------------------------------------------------------------
# Embedder — sentence-transformers wrapper
# ---------------------------------------------------------------------------

class Embedder:
    def __init__(self, model_name: str | None = None):
        self.model_name = model_name or settings.embedding_model
        self._model = None

    @property
    def model(self):
        if self._model is None:
            from sentence_transformers import SentenceTransformer
            logger.info(f"Loading embedding model: {self.model_name}")
            self._model = SentenceTransformer(self.model_name)
            logger.info("Embedding model loaded")
        return self._model

    @property
    def dimension(self) -> int:
        return self.model.get_sentence_embedding_dimension()

    def embed(self, text: str, source: str = "") -> np.ndarray:
        if not text.strip():
            return np.zeros(self.dimension)
        return self.model.encode(text, convert_to_numpy=True, normalize_embeddings=True)

    def embed_batch(self, texts: list[str], batch_size: int = 32) -> np.ndarray:
        if not texts:
            return np.zeros((0, self.dimension))
        non_empty = [(i, t) for i, t in enumerate(texts) if t.strip()]
        if not non_empty:
            return np.zeros((len(texts), self.dimension))
        indices, non_empty_texts = zip(*non_empty)
        embeddings = self.model.encode(
            list(non_empty_texts), batch_size=batch_size, convert_to_numpy=True,
            normalize_embeddings=True, show_progress_bar=len(non_empty_texts) > 100)
        result = np.zeros((len(texts), self.dimension))
        for idx, emb in zip(indices, embeddings):
            result[idx] = emb
        return result

    def similarity(self, e1: np.ndarray, e2: np.ndarray) -> float:
        return float(np.dot(e1, e2))


embedder = Embedder()


## Indexing

In [ ]:
# ---------------------------------------------------------------------------
# FAISS Vector Index
# ---------------------------------------------------------------------------
import faiss


class VectorIndex:
    def __init__(self, dimension: int | None = None):
        self.dimension = dimension or settings.embedding_dimension
        self._index = None
        self._id_to_idx: dict[str, int] = {}
        self._idx_to_id: dict[int, str] = {}
        self._next_idx = 0

    @property
    def index(self):
        if self._index is None:
            self._index = faiss.IndexFlatIP(self.dimension)
        return self._index

    def add(self, chunk_id: str, embedding: np.ndarray) -> None:
        if chunk_id in self._id_to_idx:
            self.remove([chunk_id])
        embedding = embedding.astype(np.float32).reshape(1, -1)
        self.index.add(embedding)
        self._id_to_idx[chunk_id] = self._next_idx
        self._idx_to_id[self._next_idx] = chunk_id
        self._next_idx += 1

    def add_batch(self, chunk_ids: list[str], embeddings: np.ndarray) -> None:
        existing = [c for c in chunk_ids if c in self._id_to_idx]
        if existing:
            self.remove(existing)
        embeddings = embeddings.astype(np.float32)
        if embeddings.ndim == 1:
            embeddings = embeddings.reshape(1, -1)
        start_idx = self._next_idx
        self.index.add(embeddings)
        for i, cid in enumerate(chunk_ids):
            self._id_to_idx[cid] = start_idx + i
            self._idx_to_id[start_idx + i] = cid
        self._next_idx = start_idx + len(chunk_ids)

    def search(self, query_embedding: np.ndarray, k: int = 10,
               filter_ids: list[str] | None = None) -> list[tuple[str, float]]:
        if self.index.ntotal == 0:
            return []
        query = query_embedding.astype(np.float32).reshape(1, -1)
        search_k = min(self.index.ntotal, k * 10) if filter_ids else k
        scores, indices = self.index.search(query, search_k)
        filter_set = set(filter_ids) if filter_ids else None
        results = []
        for idx, score in zip(indices[0], scores[0]):
            if idx < 0:
                continue
            cid = self._idx_to_id.get(idx)
            if cid is None:
                continue
            if filter_set and cid not in filter_set:
                continue
            results.append((cid, float(score)))
            if len(results) >= k:
                break
        return results

    def remove(self, chunk_ids: list[str]) -> int:
        removed = 0
        for cid in chunk_ids:
            if cid in self._id_to_idx:
                idx = self._id_to_idx.pop(cid)
                self._idx_to_id.pop(idx, None)
                removed += 1
        return removed

    def save(self, index_path: Path, id_map_path: Path) -> None:
        index_path.parent.mkdir(parents=True, exist_ok=True)
        faiss.write_index(self.index, str(index_path))
        with open(id_map_path, "w") as f:
            json.dump({"id_to_idx": self._id_to_idx,
                        "idx_to_id": {str(k): v for k, v in self._idx_to_id.items()},
                        "next_idx": self._next_idx}, f)

    def load(self, index_path: Path, id_map_path: Path) -> bool:
        if not index_path.exists() or not id_map_path.exists():
            return False
        try:
            self._index = faiss.read_index(str(index_path))
            with open(id_map_path) as f:
                m = json.load(f)
            self._id_to_idx = m["id_to_idx"]
            self._idx_to_id = {int(k): v for k, v in m["idx_to_id"].items()}
            self._next_idx = m["next_idx"]
            return True
        except Exception:
            self.clear()
            return False

    def clear(self) -> None:
        self._index = None
        self._id_to_idx = {}
        self._idx_to_id = {}
        self._next_idx = 0

    @property
    def size(self) -> int:
        return self.index.ntotal if self._index else 0

    @property
    def num_chunks(self) -> int:
        return len(self._id_to_idx)


### Multi-Vector Index & BM25 Keyword Index

In [ ]:
# ---------------------------------------------------------------------------
# Multi-Vector Index — 3 FAISS indices for multi-vector retrieval
# ---------------------------------------------------------------------------
from concurrent.futures import ThreadPoolExecutor


class MultiVectorIndex:
    def __init__(self, dimension: int | None = None):
        self.dimension = dimension or settings.embedding_dimension
        self.main_index = VectorIndex(self.dimension)
        self.summary_index = VectorIndex(self.dimension)
        self.question_index = VectorIndex(self.dimension)
        self._question_to_chunk: dict[str, str] = {}
        self._executor = ThreadPoolExecutor(max_workers=3)

    def add_batch(self, main_data=None, summary_data=None, question_data=None):
        if main_data:
            ids, embs = zip(*main_data)
            self.main_index.add_batch(list(ids), np.array(embs))
        if summary_data:
            ids = [f"{d[0]}_summary" for d in summary_data]
            embs = np.array([d[1] for d in summary_data])
            self.summary_index.add_batch(ids, embs)
        if question_data:
            qids = [d[0] for d in question_data]
            embs = np.array([d[2] for d in question_data])
            self.question_index.add_batch(qids, embs)
            for qid, cid, _ in question_data:
                self._question_to_chunk[qid] = cid

    def search(self, query_embedding, k=10, vector_types=None, filter_ids=None):
        vector_types = vector_types or ["main", "summary", "question"]
        results = {}
        futures = {}
        if "main" in vector_types:
            futures["main"] = self._executor.submit(self.main_index.search, query_embedding, k, filter_ids)
        if "summary" in vector_types:
            sf = [f"{c}_summary" for c in filter_ids] if filter_ids else None
            futures["summary"] = self._executor.submit(self.summary_index.search, query_embedding, k, sf)
        if "question" in vector_types:
            futures["question"] = self._executor.submit(self.question_index.search, query_embedding, k * 2, None)
        if "main" in futures:
            results["main"] = futures["main"].result()
        if "summary" in futures:
            results["summary"] = [(s.replace("_summary", ""), sc) for s, sc in futures["summary"].result()]
        if "question" in futures:
            seen = set()
            qr = []
            for qid, sc in futures["question"].result():
                cid = self._question_to_chunk.get(qid)
                if cid and cid not in seen:
                    if filter_ids is None or cid in filter_ids:
                        qr.append((cid, sc))
                        seen.add(cid)
                        if len(qr) >= k:
                            break
            results["question"] = qr
        return results

    def remove_chunk(self, chunk_id):
        self.main_index.remove([chunk_id])
        self.summary_index.remove([f"{chunk_id}_summary"])
        qids = [q for q, c in self._question_to_chunk.items() if c == chunk_id]
        if qids:
            self.question_index.remove(qids)
            for q in qids:
                del self._question_to_chunk[q]

    def save(self, base_path=None):
        bp = base_path or settings.data_folder / "faiss"
        self.main_index.save(bp / "main_index.faiss", bp / "main_id_map.json")
        self.summary_index.save(bp / "summary_index.faiss", bp / "summary_id_map.json")
        self.question_index.save(bp / "question_index.faiss", bp / "question_id_map.json")
        p = bp / "question_chunk_map.json"
        p.parent.mkdir(parents=True, exist_ok=True)
        with open(p, "w") as f:
            json.dump(self._question_to_chunk, f)

    def load(self, base_path=None):
        bp = base_path or settings.data_folder / "faiss"
        r1 = self.main_index.load(bp / "main_index.faiss", bp / "main_id_map.json")
        r2 = self.summary_index.load(bp / "summary_index.faiss", bp / "summary_id_map.json")
        r3 = self.question_index.load(bp / "question_index.faiss", bp / "question_id_map.json")
        mp = bp / "question_chunk_map.json"
        if mp.exists():
            with open(mp) as f:
                self._question_to_chunk = json.load(f)
        return r1 or r2 or r3

    def clear(self):
        self.main_index.clear()
        self.summary_index.clear()
        self.question_index.clear()
        self._question_to_chunk = {}

    @property
    def stats(self):
        return {"main": self.main_index.size, "summary": self.summary_index.size,
                "question": self.question_index.size}


# ---------------------------------------------------------------------------
# BM25 Keyword Index
# ---------------------------------------------------------------------------
from rank_bm25 import BM25Okapi


class KeywordIndex:
    def __init__(self):
        self._bm25 = None
        self._chunk_ids: list[str] = []
        self._corpus: list[list[str]] = []

    def add(self, chunk_id: str, text: str):
        tokens = self._tokenize(text)
        if chunk_id in self._chunk_ids:
            self._corpus[self._chunk_ids.index(chunk_id)] = tokens
        else:
            self._chunk_ids.append(chunk_id)
            self._corpus.append(tokens)
        self._bm25 = None

    def add_batch(self, chunk_ids, texts):
        for cid, text in zip(chunk_ids, texts):
            self.add(cid, text)

    def search(self, query, k=10, expand_query=None):
        if not self._corpus:
            return []
        self._ensure_bm25()
        should_expand = expand_query if expand_query is not None else settings.query_expand_synonyms
        tokens = self._tokenize(query, expand_synonyms=should_expand)
        if not tokens:
            return []
        scores = self._bm25.get_scores(tokens)
        ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
        return [(self._chunk_ids[i], float(s)) for i, s in ranked[:k] if s > 0]

    def search_with_keywords(self, query, k=10, expand_query=None):
        if not self._corpus:
            return []
        self._ensure_bm25()
        should_expand = expand_query if expand_query is not None else settings.query_expand_synonyms
        tokens = self._tokenize(query, expand_synonyms=should_expand)
        if not tokens:
            return []
        scores = self._bm25.get_scores(tokens)
        ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
        results = []
        for i, s in ranked[:k]:
            if s > 0:
                matched = [t for t in tokens if t in set(self._corpus[i])]
                results.append((self._chunk_ids[i], float(s), matched))
        return results

    def remove(self, chunk_ids):
        removed = 0
        for cid in chunk_ids:
            if cid in self._chunk_ids:
                idx = self._chunk_ids.index(cid)
                self._chunk_ids.pop(idx)
                self._corpus.pop(idx)
                removed += 1
        if removed:
            self._bm25 = None
        return removed

    def save(self, path=None):
        path = path or settings.bm25_index_path
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "wb") as f:
            pickle.dump({"chunk_ids": self._chunk_ids, "corpus": self._corpus}, f)

    def load(self, path=None):
        path = path or settings.bm25_index_path
        if not path.exists():
            return False
        with open(path, "rb") as f:
            data = pickle.load(f)
        self._chunk_ids = data["chunk_ids"]
        self._corpus = data["corpus"]
        self._bm25 = None
        return True

    def clear(self):
        self._chunk_ids, self._corpus, self._bm25 = [], [], None

    def _tokenize(self, text, expand_synonyms=None):
        text = re.sub(r"[^\w\s]", " ", text.lower())
        should_expand = expand_synonyms if expand_synonyms is not None else settings.bm25_expand_synonyms
        processed = []
        for token in text.split():
            if len(token) < settings.bm25_min_token_length or token in STOPWORDS:
                continue
            if should_expand and token in INSURANCE_ACRONYMS:
                expanded = INSURANCE_ACRONYMS[token]
                processed.extend(t for t in expanded.split()
                                 if len(t) >= settings.bm25_min_token_length and t not in STOPWORDS)
                continue
            stemmed = self._simple_stem(token)
            processed.append(stemmed)
            if should_expand and stemmed in INSURANCE_SYNONYMS:
                for syn in INSURANCE_SYNONYMS[stemmed]:
                    ss = self._simple_stem(syn)
                    if ss not in processed:
                        processed.append(ss)
        return processed

    def _simple_stem(self, word):
        for sfx, rep in [("ational","ate"),("tional","tion"),("encies","ence"),("ancies","ance"),
                         ("iveness","ive"),("fulness","ful"),("ousness","ous"),("ization","ize"),
                         ("isation","ise"),("ating","ate"),("izing","ize"),("ising","ise"),
                         ("ities","ity"),("ments","ment"),("ness",""),("ings",""),("tion","t"),
                         ("sion","s"),("ious",""),("eous",""),("ment",""),("able",""),("ible",""),
                         ("ally",""),("ful",""),("ous",""),("ive",""),("ing",""),("ion",""),
                         ("ies","y"),("es",""),("ed",""),("ly",""),("er",""),("s","")]:
            if word.endswith(sfx) and len(word) > len(sfx) + 2:
                return word[:-len(sfx)] + rep
        return word

    def _ensure_bm25(self):
        if self._bm25 is None and self._corpus:
            self._bm25 = BM25Okapi(self._corpus)

    @property
    def size(self):
        return len(self._chunk_ids)


### SQLite Metadata Store

In [ ]:
# ---------------------------------------------------------------------------
# SQLite Metadata Store
# CRITICAL: For :memory: databases, _write_db() reuses the persistent
# read connection (each aiosqlite.connect(":memory:") creates a separate DB).
# ---------------------------------------------------------------------------

class MetadataStore:
    def __init__(self, db_path=None):
        raw = db_path or settings.database_path
        self.db_path = raw if isinstance(raw, str) and raw == ":memory:" else (Path(raw) if not isinstance(raw, Path) else raw)
        self._db: aiosqlite.Connection | None = None

    async def _get_db(self):
        if self._db is None:
            self._db = await aiosqlite.connect(self.db_path if isinstance(self.db_path, str) else str(self.db_path))
            self._db.row_factory = aiosqlite.Row
            await self._db.execute("PRAGMA foreign_keys = ON")
        return self._db

    @asynccontextmanager
    async def _write_db(self):
        if isinstance(self.db_path, str) and self.db_path == ":memory:":
            yield await self._get_db()
        else:
            async with aiosqlite.connect(str(self.db_path)) as db:
                await db.execute("PRAGMA foreign_keys = ON")
                yield db

    async def close(self):
        if self._db:
            await self._db.close()
            self._db = None

    async def initialize(self):
        if isinstance(self.db_path, Path):
            self.db_path.parent.mkdir(parents=True, exist_ok=True)
        async with self._write_db() as db:
            await db.execute("PRAGMA journal_mode=WAL")
            for sql in [
                """CREATE TABLE IF NOT EXISTS documents (
                    id TEXT PRIMARY KEY, file_path TEXT UNIQUE NOT NULL, file_name TEXT NOT NULL,
                    file_type TEXT NOT NULL, file_hash TEXT NOT NULL,
                    detected_doc_type TEXT DEFAULT 'unknown', summary TEXT DEFAULT '',
                    entities TEXT DEFAULT '{}', key_topics TEXT DEFAULT '[]',
                    table_descriptions TEXT DEFAULT '[]', indexed_at TEXT NOT NULL,
                    sheet_names TEXT, column_schema TEXT, row_count INTEGER, date_range TEXT,
                    processing_status TEXT DEFAULT 'pending', layout_data TEXT,
                    extracted_markdown TEXT, reviewed_markdown TEXT, page_count INTEGER)""",
                """CREATE TABLE IF NOT EXISTS document_entities (
                    id INTEGER PRIMARY KEY AUTOINCREMENT, document_id TEXT NOT NULL,
                    entity_key TEXT NOT NULL, entity_value TEXT NOT NULL,
                    FOREIGN KEY (document_id) REFERENCES documents(id) ON DELETE CASCADE)""",
                """CREATE TABLE IF NOT EXISTS chunks (
                    id TEXT PRIMARY KEY, document_id TEXT NOT NULL, text TEXT NOT NULL,
                    contextualized_text TEXT NOT NULL, content_type TEXT DEFAULT 'paragraph',
                    page INTEGER, sheet_name TEXT, heading_path TEXT, entities TEXT DEFAULT '{}',
                    chunk_index INTEGER DEFAULT 0, parent_chunk_id TEXT,
                    hierarchy_level INTEGER DEFAULT 0, bbox TEXT, layout_label TEXT,
                    is_semantic_boundary INTEGER DEFAULT 0, semantic_similarity_prev REAL,
                    FOREIGN KEY (document_id) REFERENCES documents(id) ON DELETE CASCADE)""",
                """CREATE TABLE IF NOT EXISTS chunk_metadata (
                    chunk_id TEXT PRIMARY KEY, title TEXT, summary TEXT, keywords TEXT,
                    entities TEXT, category TEXT, contextual_description TEXT,
                    temporal_context TEXT, enriched_at TEXT,
                    FOREIGN KEY (chunk_id) REFERENCES chunks(id) ON DELETE CASCADE)""",
                """CREATE TABLE IF NOT EXISTS chunk_questions (
                    id INTEGER PRIMARY KEY AUTOINCREMENT, chunk_id TEXT NOT NULL,
                    question TEXT NOT NULL, vector_id TEXT,
                    FOREIGN KEY (chunk_id) REFERENCES chunks(id) ON DELETE CASCADE)""",
                """CREATE TABLE IF NOT EXISTS vector_embeddings (
                    id TEXT PRIMARY KEY, chunk_id TEXT NOT NULL, vector_type TEXT NOT NULL,
                    source_text TEXT, question_id INTEGER,
                    FOREIGN KEY (chunk_id) REFERENCES chunks(id) ON DELETE CASCADE)""",
            ]:
                await db.execute(sql)
            for idx in [
                "CREATE INDEX IF NOT EXISTS idx_doc_path ON documents(file_path)",
                "CREATE INDEX IF NOT EXISTS idx_doc_type ON documents(detected_doc_type)",
                "CREATE INDEX IF NOT EXISTS idx_ent_key ON document_entities(entity_key)",
                "CREATE INDEX IF NOT EXISTS idx_chunks_doc ON chunks(document_id)",
                "CREATE INDEX IF NOT EXISTS idx_chunks_parent ON chunks(parent_chunk_id)",
                "CREATE INDEX IF NOT EXISTS idx_meta_chunk ON chunk_metadata(chunk_id)",
                "CREATE INDEX IF NOT EXISTS idx_q_chunk ON chunk_questions(chunk_id)",
                "CREATE INDEX IF NOT EXISTS idx_vec_chunk ON vector_embeddings(chunk_id)",
            ]:
                await db.execute(idx)
            await db.commit()

    # --- Document CRUD ---
    async def add_document(self, doc: Document):
        async with self._write_db() as db:
            await db.execute(
                """INSERT OR REPLACE INTO documents
                (id,file_path,file_name,file_type,file_hash,detected_doc_type,summary,entities,
                 key_topics,table_descriptions,indexed_at,sheet_names,column_schema,row_count,date_range)
                VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)""",
                (doc.id, doc.file_path, doc.file_name, doc.file_type, doc.file_hash,
                 doc.detected_doc_type, doc.summary, json.dumps(doc.entities),
                 json.dumps(doc.key_topics), json.dumps(doc.table_descriptions),
                 doc.indexed_at.isoformat(),
                 json.dumps(doc.sheet_names) if doc.sheet_names else None,
                 json.dumps(doc.column_schema) if doc.column_schema else None,
                 doc.row_count, doc.date_range))
            await db.execute("DELETE FROM document_entities WHERE document_id=?", (doc.id,))
            for k, v in doc.entities.items():
                await db.execute("INSERT INTO document_entities (document_id,entity_key,entity_value) VALUES (?,?,?)",
                                 (doc.id, k, str(v)))
            await db.commit()

    async def get_document(self, document_id):
        db = await self._get_db()
        cur = await db.execute("SELECT * FROM documents WHERE id=?", (document_id,))
        row = await cur.fetchone()
        return self._row_to_document(row) if row else None

    async def get_documents_by_ids(self, ids):
        if not ids:
            return []
        ph = ",".join("?" * len(ids))
        db = await self._get_db()
        cur = await db.execute(f"SELECT * FROM documents WHERE id IN ({ph})", ids)
        return [self._row_to_document(r) for r in await cur.fetchall()]

    async def get_all_documents(self, skip=0, limit=100):
        db = await self._get_db()
        cur = await db.execute(
            "SELECT id,file_name,file_type,detected_doc_type,summary,indexed_at FROM documents ORDER BY indexed_at DESC LIMIT ? OFFSET ?",
            (limit, skip))
        return [DocumentSummary(id=r["id"], file_name=r["file_name"], file_type=r["file_type"],
                detected_doc_type=r["detected_doc_type"], summary=r["summary"],
                indexed_at=datetime.fromisoformat(r["indexed_at"])) for r in await cur.fetchall()]

    async def delete_document(self, document_id):
        async with self._write_db() as db:
            cur = await db.execute("DELETE FROM documents WHERE id=?", (document_id,))
            await db.commit()
            return cur.rowcount > 0

    async def update_document_status(self, document_id, status):
        async with self._write_db() as db:
            await db.execute("UPDATE documents SET processing_status=? WHERE id=?", (status, document_id))
            await db.commit()

    async def update_document_markdown(self, document_id, md_text):
        async with self._write_db() as db:
            await db.execute("UPDATE documents SET extracted_markdown=? WHERE id=?", (md_text, document_id))
            await db.commit()

    async def update_document_enrichment(self, document_id, doc_type, summary, entities, topics, tables):
        async with self._write_db() as db:
            await db.execute(
                "UPDATE documents SET detected_doc_type=?,summary=?,entities=?,key_topics=?,table_descriptions=?,processing_status='indexed' WHERE id=?",
                (doc_type, summary, json.dumps(entities), json.dumps(topics), json.dumps(tables), document_id))
            await db.commit()

    async def get_document_markdown(self, document_id):
        db = await self._get_db()
        cur = await db.execute("SELECT extracted_markdown,reviewed_markdown,processing_status FROM documents WHERE id=?", (document_id,))
        row = await cur.fetchone()
        return {"extracted_markdown": row[0], "reviewed_markdown": row[1], "processing_status": row[2]} if row else None

    async def filter_documents_by_type(self, doc_type):
        db = await self._get_db()
        cur = await db.execute("SELECT id FROM documents WHERE detected_doc_type LIKE ?", (f"%{doc_type}%",))
        return [r[0] for r in await cur.fetchall()]

    async def filter_documents_by_entity(self, key, value=None):
        db = await self._get_db()
        if value:
            cur = await db.execute("SELECT DISTINCT document_id FROM document_entities WHERE entity_key=? AND entity_value LIKE ?", (key, f"%{value}%"))
        else:
            cur = await db.execute("SELECT DISTINCT document_id FROM document_entities WHERE entity_key=?", (key,))
        return [r[0] for r in await cur.fetchall()]

    async def clear_all(self):
        async with self._write_db() as db:
            for t in ["vector_embeddings","chunk_questions","chunk_metadata","chunks","document_entities","documents"]:
                await db.execute(f"DELETE FROM {t}")
            await db.commit()

    # --- Chunk CRUD ---
    async def add_chunks(self, chunks):
        if not chunks:
            return
        async with self._write_db() as db:
            await db.executemany(
                """INSERT OR REPLACE INTO chunks
                (id,document_id,text,contextualized_text,content_type,page,sheet_name,heading_path,
                 entities,chunk_index,parent_chunk_id,hierarchy_level,bbox,layout_label,
                 is_semantic_boundary,semantic_similarity_prev) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)""",
                [(c.id, c.document_id, c.text, c.contextualized_text, c.content_type, c.page,
                  c.sheet_name, c.heading_path, json.dumps(c.entities), c.chunk_index,
                  c.parent_chunk_id, c.hierarchy_level,
                  json.dumps(c.bbox) if c.bbox else None, c.layout_label,
                  1 if c.is_semantic_boundary else 0, c.semantic_similarity_prev) for c in chunks])
            await db.commit()

    async def get_chunks_by_document(self, document_id):
        db = await self._get_db()
        cur = await db.execute("SELECT * FROM chunks WHERE document_id=? ORDER BY chunk_index", (document_id,))
        return [self._row_to_chunk(r) for r in await cur.fetchall()]

    async def get_chunk(self, chunk_id):
        db = await self._get_db()
        cur = await db.execute("SELECT * FROM chunks WHERE id=?", (chunk_id,))
        row = await cur.fetchone()
        return self._row_to_chunk(row) if row else None

    async def get_chunks_by_ids(self, chunk_ids):
        if not chunk_ids:
            return []
        ph = ",".join("?" * len(chunk_ids))
        db = await self._get_db()
        cur = await db.execute(f"SELECT * FROM chunks WHERE id IN ({ph})", chunk_ids)
        return [self._row_to_chunk(r) for r in await cur.fetchall()]

    async def delete_chunks_by_document(self, document_id):
        async with self._write_db() as db:
            cur = await db.execute("DELETE FROM chunks WHERE document_id=?", (document_id,))
            await db.commit()
            return cur.rowcount

    # --- Chunk Metadata ---
    async def add_chunk_metadata(self, m: ChunkMetadata):
        async with self._write_db() as db:
            await db.execute(
                "INSERT OR REPLACE INTO chunk_metadata (chunk_id,title,summary,keywords,entities,category,contextual_description,temporal_context,enriched_at) VALUES (?,?,?,?,?,?,?,?,?)",
                (m.chunk_id, m.title, m.summary, json.dumps(m.keywords), json.dumps(m.entities),
                 m.category, m.contextual_description, m.temporal_context,
                 m.enriched_at.isoformat() if m.enriched_at else datetime.utcnow().isoformat()))
            await db.commit()

    async def get_chunk_metadata_batch(self, chunk_ids):
        if not chunk_ids:
            return {}
        ph = ",".join("?" * len(chunk_ids))
        db = await self._get_db()
        cur = await db.execute(f"SELECT * FROM chunk_metadata WHERE chunk_id IN ({ph})", chunk_ids)
        return {r["chunk_id"]: self._row_to_chunk_metadata(r) for r in await cur.fetchall()}

    async def filter_chunks_by_entity(self, key, value=None):
        db = await self._get_db()
        if value:
            cur = await db.execute("SELECT chunk_id FROM chunk_metadata WHERE json_extract(entities,?)IS NOT NULL AND entities LIKE ?",
                                   (f"$.{key}", f"%{value}%"))
        else:
            cur = await db.execute("SELECT chunk_id FROM chunk_metadata WHERE json_extract(entities,?)IS NOT NULL AND json_array_length(json_extract(entities,?))>0",
                                   (f"$.{key}", f"$.{key}"))
        return [r["chunk_id"] for r in await cur.fetchall()]

    # --- Chunk Questions ---
    async def add_chunk_questions(self, chunk_id, questions):
        if not questions:
            return
        async with self._write_db() as db:
            await db.execute("DELETE FROM chunk_questions WHERE chunk_id=?", (chunk_id,))
            await db.executemany("INSERT INTO chunk_questions (chunk_id,question,vector_id) VALUES (?,?,?)",
                                 [(chunk_id, q.question, q.vector_id) for q in questions])
            await db.commit()

    async def get_chunk_questions_batch(self, chunk_ids):
        if not chunk_ids:
            return {}
        ph = ",".join("?" * len(chunk_ids))
        db = await self._get_db()
        cur = await db.execute(f"SELECT * FROM chunk_questions WHERE chunk_id IN ({ph})", chunk_ids)
        result = {c: [] for c in chunk_ids}
        for r in await cur.fetchall():
            result[r["chunk_id"]].append(ChunkQuestion(id=r["id"], chunk_id=r["chunk_id"],
                                                        question=r["question"], vector_id=r["vector_id"]))
        return result

    async def update_question_vector_id(self, qid, vid):
        async with self._write_db() as db:
            await db.execute("UPDATE chunk_questions SET vector_id=? WHERE id=?", (vid, qid))
            await db.commit()

    # --- Vector Embeddings ---
    async def add_vector_embeddings_batch(self, embeddings):
        if not embeddings:
            return
        async with self._write_db() as db:
            await db.executemany(
                "INSERT OR REPLACE INTO vector_embeddings (id,chunk_id,vector_type,source_text,question_id) VALUES (?,?,?,?,?)",
                [(e.id, e.chunk_id, e.vector_type, e.source_text, e.question_id) for e in embeddings])
            await db.commit()

    async def update_chunks_contextualized_text(self, chunks):
        if not chunks:
            return
        async with self._write_db() as db:
            await db.executemany("UPDATE chunks SET contextualized_text=? WHERE id=?",
                                 [(c.contextualized_text, c.id) for c in chunks])
            await db.commit()

    async def get_statistics(self):
        db = await self._get_db()
        docs = (await (await db.execute("SELECT COUNT(*) FROM documents")).fetchone())[0]
        chunks = (await (await db.execute("SELECT COUNT(*) FROM chunks")).fetchone())[0]
        return {"total_documents": docs, "total_chunks": chunks}

    # --- Row converters ---
    def _row_to_document(self, row):
        rk = row.keys() if hasattr(row, "keys") else []
        return Document(
            id=row["id"], file_path=row["file_path"], file_name=row["file_name"],
            file_type=row["file_type"], file_hash=row["file_hash"],
            detected_doc_type=row["detected_doc_type"] or "unknown",
            summary=row["summary"] or "",
            entities=json.loads(row["entities"]) if row["entities"] else {},
            key_topics=json.loads(row["key_topics"]) if row["key_topics"] else [],
            table_descriptions=json.loads(row["table_descriptions"]) if row["table_descriptions"] else [],
            indexed_at=datetime.fromisoformat(row["indexed_at"]),
            sheet_names=json.loads(row["sheet_names"]) if row["sheet_names"] else None,
            column_schema=json.loads(row["column_schema"]) if row["column_schema"] else None,
            row_count=row["row_count"],
            date_range=row["date_range"] if "date_range" in rk else None,
            processing_status=row["processing_status"] if "processing_status" in rk else "pending",
            layout_data=row["layout_data"] if "layout_data" in rk else None,
            extracted_markdown=row["extracted_markdown"] if "extracted_markdown" in rk else None,
            reviewed_markdown=row["reviewed_markdown"] if "reviewed_markdown" in rk else None,
            page_count=row["page_count"] if "page_count" in rk else None)

    def _row_to_chunk(self, row):
        rk = row.keys() if hasattr(row, "keys") else []
        return Chunk(
            id=row["id"], document_id=row["document_id"], text=row["text"],
            contextualized_text=row["contextualized_text"], content_type=row["content_type"],
            page=row["page"], sheet_name=row["sheet_name"], heading_path=row["heading_path"],
            entities=json.loads(row["entities"]), chunk_index=row["chunk_index"],
            parent_chunk_id=row["parent_chunk_id"] if "parent_chunk_id" in rk else None,
            hierarchy_level=row["hierarchy_level"] if "hierarchy_level" in rk else 0,
            bbox=json.loads(row["bbox"]) if "bbox" in rk and row["bbox"] else None,
            layout_label=row["layout_label"] if "layout_label" in rk else None,
            is_semantic_boundary=bool(row["is_semantic_boundary"]) if "is_semantic_boundary" in rk else False,
            semantic_similarity_prev=row["semantic_similarity_prev"] if "semantic_similarity_prev" in rk else None)

    def _row_to_chunk_metadata(self, row):
        rk = row.keys() if hasattr(row, "keys") else []
        return ChunkMetadata(
            chunk_id=row["chunk_id"], title=row["title"], summary=row["summary"],
            keywords=json.loads(row["keywords"]) if row["keywords"] else [],
            entities=json.loads(row["entities"]) if row["entities"] else {},
            category=row["category"], contextual_description=row["contextual_description"],
            temporal_context=row["temporal_context"] if "temporal_context" in rk else None,
            enriched_at=datetime.fromisoformat(row["enriched_at"]) if row["enriched_at"] else None)


## Search

In [ ]:
# ---------------------------------------------------------------------------
# Query Processor — heuristic query understanding
# ---------------------------------------------------------------------------

class QueryProcessor:
    def __init__(self, use_llm: bool = False):
        self.use_llm = use_llm
        self._llm_client = None

    async def process(self, query: str) -> QueryIntent:
        q = query.lower().strip()
        doc_type = self._detect_doc_type(q)
        needs_syn = self._detect_synthesis_need(q)
        qtype = self._detect_query_type(q)
        cats = self._detect_preferred_categories(q)
        if self.use_llm and needs_syn:
            try:
                return await self._llm_understand(query)
            except Exception as e:
                logger.warning(f"LLM query understanding failed: {e}")
        return QueryIntent(search_terms=query, doc_type_filter=doc_type, entity_filters={},
                           needs_synthesis=needs_syn, query_type=qtype, preferred_categories=cats)

    def _detect_doc_type(self, q):
        for dt, kws in DOC_TYPE_KEYWORDS.items():
            for kw in kws:
                if kw in q:
                    return dt
        return None

    def _detect_synthesis_need(self, q):
        for kw in SYNTHESIS_KEYWORDS:
            if kw in q:
                return True
        return q.endswith("?") or len(q.split()) > 10

    def _detect_preferred_categories(self, q):
        cats = []
        for cat, kws in CATEGORY_KEYWORDS.items():
            for kw in kws:
                if kw in q:
                    cats.append(cat)
                    break
        return cats

    def _detect_query_type(self, q):
        if any(w in q for w in ["compare", "difference", "versus", "vs"]):
            return "comparative"
        if any(w in q for w in ["total", "sum", "average", "count", "how many"]):
            return "aggregation"
        if any(w in q for w in ["what", "how", "why", "explain"]):
            return "exploratory"
        return "factual"

    async def _llm_understand(self, query):
        if self._llm_client is None:
            self._llm_client = get_llm_client()
        result = await self._llm_client.complete_json(QUERY_UNDERSTANDING_PROMPT.format(query=query))
        if not result:
            return QueryIntent(search_terms=query)
        return QueryIntent(search_terms=result.get("search_terms", query),
                           doc_type_filter=result.get("doc_type_filter"),
                           entity_filters=result.get("entity_filters", {}),
                           needs_synthesis=result.get("needs_synthesis", False),
                           query_type=result.get("query_type", "factual"))


query_processor = QueryProcessor()


# ---------------------------------------------------------------------------
# HyDE — Hypothetical Document Embeddings
# ---------------------------------------------------------------------------

HYDE_SYSTEM_PROMPT = """You are a helpful assistant that generates detailed, factual responses.
Write a comprehensive paragraph answering the given question.
Focus on specific, informative content. Write as if stating facts from a document."""

HYDE_USER_PROMPT = """Question: {query}

Write a detailed paragraph (150-250 words) answering this question as it would appear in a relevant document."""


class HyDEQueryExpander:
    def __init__(self):
        self._llm_client = None

    @property
    def llm_client(self):
        if self._llm_client is None:
            self._llm_client = get_llm_client()
        return self._llm_client

    async def generate_hypothetical(self, query):
        try:
            resp = await self.llm_client.complete(HYDE_USER_PROMPT.format(query=query),
                                                   system_prompt=HYDE_SYSTEM_PROMPT, max_tokens=500, temperature=0.7)
            return resp.strip() if resp and len(resp.strip()) > 50 else None
        except Exception as e:
            logger.error(f"HyDE failed: {e}")
            return None

    def get_hyde_embedding(self, query, hypothetical, alpha=0.5, query_embedding=None):
        qe = query_embedding if query_embedding is not None else embedder.embed(query)
        he = embedder.embed(hypothetical)
        combined = (1 - alpha) * qe + alpha * he
        norm = np.linalg.norm(combined)
        return combined / norm if norm > 0 else combined

    def _adaptive_alpha(self, query):
        words = len(query.split())
        if words <= 3:
            return settings.hyde_alpha_min
        if query.endswith("?") and words <= 8:
            return (settings.hyde_alpha_min + settings.hyde_alpha_max) / 2
        if words <= 10:
            return settings.hyde_alpha_max * 0.8
        return settings.hyde_alpha_max

    async def expand_query(self, query, query_embedding=None):
        if query_embedding is None:
            query_embedding = embedder.embed(query)
        if not settings.enable_hyde:
            return query_embedding, None
        hypo = await self.generate_hypothetical(query)
        if hypo:
            alpha = self._adaptive_alpha(query) if settings.hyde_adaptive else settings.hyde_alpha
            return self.get_hyde_embedding(query, hypo, alpha, query_embedding), hypo
        return query_embedding, None


hyde_expander = HyDEQueryExpander()


### Enhanced Hybrid Search

In [ ]:
# ---------------------------------------------------------------------------
# Enhanced Hybrid Search — multi-source RRF fusion retrieval
# ---------------------------------------------------------------------------

class EnhancedHybridSearch:
    def __init__(self, weights=None, k_constant=None):
        self.weights = weights or settings.multi_vector_weights
        self.k_constant = k_constant if k_constant is not None else settings.rrf_k_constant

    async def search(self, query, k=10, filters=None, document_id=None,
                     query_type=None, query_embedding=None, preferred_categories=None):
        if query_type and self.weights == settings.multi_vector_weights:
            self.weights = settings.get_weight_profile(query_type)
        if query_embedding is None:
            query_embedding = embedder.embed(query)

        # Filters
        filter_cids = None
        if document_id:
            filter_cids = [c.id for c in await metadata_store.get_chunks_by_document(document_id)]
        elif filters:
            filter_cids = await self._apply_filters(filters)

        search_k = k * 3

        # Parallel BM25 + HyDE
        async def _bm25():
            return keyword_index.search_with_keywords(query, k=search_k)
        async def _hyde():
            return await hyde_expander.expand_query(query, query_embedding) if settings.enable_hyde else (query_embedding, None)

        bm25_task = asyncio.create_task(_bm25())
        hyde_task = asyncio.create_task(_hyde())
        final_emb, _ = await hyde_task

        # FAISS search
        vr = multi_vector_index.search(final_emb, k=search_k, filter_ids=filter_cids)
        main_r, sum_r, q_r = vr.get("main", []), vr.get("summary", []), vr.get("question", [])

        bm25_raw = await bm25_task
        bm25_r = [(c, s) for c, s, _ in bm25_raw]
        if filter_cids:
            fs = set(filter_cids)
            bm25_r = [(c, s) for c, s in bm25_r if c in fs]

        # RRF
        fused, _ = self._rrf(main_r, sum_r, q_r, bm25_r)
        if preferred_categories:
            fused = await self._category_boost(fused, preferred_categories)

        top = fused[:k]

        # Build results
        top_ids = [c for c, _ in top]
        scores = {c: s for c, s in top}
        chunks = {c.id: c for c in await metadata_store.get_chunks_by_ids(top_ids)}
        docs = {d.id: d for d in await metadata_store.get_documents_by_ids(
            list(set(c.document_id for c in chunks.values())))}
        meta = await metadata_store.get_chunk_metadata_batch(top_ids)

        results = []
        for cid in top_ids:
            ch = chunks.get(cid)
            if not ch:
                continue
            doc = docs.get(ch.document_id)
            if not doc:
                continue
            m = meta.get(cid)
            results.append(SearchResultItem(
                document_id=doc.id, file_name=doc.file_name, file_type=doc.file_type,
                detected_doc_type=doc.detected_doc_type, chunk_text=ch.text[:500],
                chunk_id=cid, score=scores[cid], page=ch.page, sheet_name=ch.sheet_name,
                heading_path=ch.heading_path, highlights=self._highlights(ch.text, query),
                entities=ch.entities, temporal_context=m.temporal_context if m else None,
                chunk_title=m.title if m else None, chunk_keywords=m.keywords if m else []))
        return results

    def _rrf(self, main, summary, question, bm25):
        scores, details = {}, {}
        for name, results, weight in [
            ("main_vector", main, self.weights.get("main_vector", 0.35)),
            ("summary_vector", summary, self.weights.get("summary_vector", 0.15)),
            ("question_vector", question, self.weights.get("question_vector", 0.25)),
            ("bm25", bm25, self.weights.get("bm25", 0.25)),
        ]:
            for rank, (cid, _) in enumerate(results, 1):
                scores.setdefault(cid, 0)
                details.setdefault(cid, {})
                rrf = weight / (self.k_constant + rank)
                scores[cid] += rrf
                details[cid][name] = rrf
        return sorted(scores.items(), key=lambda x: x[1], reverse=True), details

    async def _category_boost(self, fused, cats, factor=1.3):
        ids = [c for c, _ in fused[:50]]
        meta = await metadata_store.get_chunk_metadata_batch(ids)
        cs = set(cats)
        boosted = [(c, s * factor if meta.get(c) and meta[c].category in cs else s) for c, s in fused]
        boosted.sort(key=lambda x: x[1], reverse=True)
        return boosted

    async def _apply_filters(self, filters):
        cids = None
        if "doc_type" in filters and filters["doc_type"]:
            cids = set(await metadata_store.filter_documents_by_type(filters["doc_type"]))
        if "entities" in filters:
            for k, v in filters["entities"].items():
                eids = set(await metadata_store.filter_documents_by_entity(k, v))
                cids = eids if cids is None else cids & eids
        if cids is None:
            return None
        all_cids = []
        for did in cids:
            all_cids.extend(c.id for c in await metadata_store.get_chunks_by_document(did))
        return all_cids or None

    def _highlights(self, text, query):
        hl, tl = [], text.lower()
        for t in query.lower().split():
            if len(t) < 2 or t not in tl:
                continue
            idx = tl.find(t)
            s, e = max(0, idx - 50), min(len(text), idx + len(t) + 50)
            snip = ("..." if s > 0 else "") + text[s:e] + ("..." if e < len(text) else "")
            hl.append(snip)
            if len(hl) >= 3:
                break
        return hl


### MMR & Semantic Cache

In [ ]:
# ---------------------------------------------------------------------------
# MMR — Maximum Marginal Relevance
# ---------------------------------------------------------------------------

class MMRSelector:
    def __init__(self, lambda_param=None):
        self.lambda_param = lambda_param if lambda_param is not None else settings.mmr_lambda

    def select(self, query_embedding, results, k, lambda_param=None):
        if not results or len(results) <= k:
            return [(c, s) for c, s, _ in results] if results else []
        lv = lambda_param if lambda_param is not None else self.lambda_param
        texts = [t for _, _, t in results]
        doc_embs = embedder.embed_batch(texts)
        qn = query_embedding / (np.linalg.norm(query_embedding) + 1e-9)
        dn = doc_embs / (np.linalg.norm(doc_embs, axis=1, keepdims=True) + 1e-9)
        qsims = np.dot(dn, qn)
        selected, remaining = [], list(range(len(results)))
        for _ in range(k):
            if not remaining:
                break
            best_s, best_i = float("-inf"), remaining[0]
            for i in remaining:
                rel = qsims[i]
                div = float(np.max(np.dot(dn[selected], dn[i]))) if selected else 0.0
                ms = lv * rel - (1 - lv) * div
                if ms > best_s:
                    best_s, best_i = ms, i
            selected.append(best_i)
            remaining.remove(best_i)
        return [(results[i][0], 1.0 - (j / len(selected)) * 0.5) for j, i in enumerate(selected)]


mmr_selector = MMRSelector()


# ---------------------------------------------------------------------------
# Semantic Cache — LRU with vectorized similarity
# ---------------------------------------------------------------------------

class SemanticCache:
    def __init__(self, threshold=None, ttl=None, max_entries=None):
        self.threshold = threshold or settings.cache_similarity_threshold
        self.ttl = ttl or settings.cache_ttl_seconds
        self.max_entries = max_entries or settings.cache_max_entries
        self._cache: OrderedDict = OrderedDict()
        self._embeddings: dict[str, np.ndarray] = {}
        self._matrix = None
        self._keys = []
        self._dirty = True

    async def get(self, query, query_embedding=None):
        if not self._cache:
            return None
        if query_embedding is None:
            query_embedding = embedder.embed(query)
        if self._dirty:
            self._rebuild()
        if self._matrix is None:
            return None
        scores = self._matrix @ query_embedding
        bi = int(np.argmax(scores))
        if float(scores[bi]) < self.threshold:
            return None
        key = self._keys[bi]
        cached = self._cache.get(key)
        if not cached or time.time() - cached["created_at"] > self.ttl:
            if cached:
                self._remove(key)
            return None
        self._cache.move_to_end(key)
        return cached

    async def set(self, query, response, document_ids, query_embedding=None):
        key = hashlib.sha256(query.lower().strip().encode()).hexdigest()[:16]
        if query_embedding is None:
            query_embedding = embedder.embed(query)
        while len(self._cache) >= self.max_entries:
            ok, _ = self._cache.popitem(last=False)
            self._embeddings.pop(ok, None)
            self._dirty = True
        self._cache[key] = {"query": query, "response": response,
                             "document_ids": document_ids, "created_at": time.time()}
        self._embeddings[key] = query_embedding
        self._dirty = True

    def invalidate_for_document(self, doc_id):
        rm = [k for k, v in self._cache.items() if doc_id in v.get("document_ids", [])]
        for k in rm:
            self._remove(k)
        return len(rm)

    def clear(self):
        self._cache.clear()
        self._embeddings.clear()
        self._matrix = None
        self._keys = []
        self._dirty = True

    def _rebuild(self):
        if self._embeddings:
            self._keys = list(self._embeddings.keys())
            self._matrix = np.stack([self._embeddings[k] for k in self._keys])
        else:
            self._matrix, self._keys = None, []
        self._dirty = False

    def _remove(self, key):
        self._cache.pop(key, None)
        self._embeddings.pop(key, None)
        self._dirty = True


semantic_cache = SemanticCache()


## Response Generator

In [ ]:
# ---------------------------------------------------------------------------
# Response Generator — end-to-end search + optional LLM synthesis
# ---------------------------------------------------------------------------

class ResponseGenerator:
    def __init__(self):
        self._llm_client = None

    async def generate(self, query, mode="auto", k=10, filters=None):
        t0 = time.time()
        qe = embedder.embed(query)

        # Cache check
        cached = await semantic_cache.get(query, query_embedding=qe)
        if cached:
            ms = (time.time() - t0) * 1000
            r = cached["response"]
            r["latency_ms"] = ms
            r["cache_hit"] = True
            r["response_tier"] = "cache"
            return SearchResponse(**r)

        # Query intent
        intent = await query_processor.process(query)
        filt = filters or {}
        if intent.doc_type_filter:
            filt["doc_type"] = intent.doc_type_filter
        if intent.entity_filters:
            filt["entities"] = intent.entity_filters

        # Search
        results = await enhanced_hybrid_search.search(
            query=query, k=k, filters=filt or None, query_embedding=qe,
            preferred_categories=intent.preferred_categories or None)

        if mode == "auto":
            mode = "synthesis" if intent.needs_synthesis else "retrieval"

        answer = None
        tier = "retrieval"
        if mode == "synthesis" and results:
            answer = await self._synthesize(query, results)
            tier = "synthesis"

        sources = self._sources(results)
        ms = (time.time() - t0) * 1000

        resp = SearchResponse(query=query, results=results, answer=answer,
                              total_results=len(results), latency_ms=ms,
                              cache_hit=False, response_tier=tier, sources=sources)

        doc_ids = list(set(r.document_id for r in results))
        await semantic_cache.set(query, resp.model_dump(), doc_ids, qe)
        return resp

    async def _synthesize(self, query, results):
        if self._llm_client is None:
            self._llm_client = get_synthesis_llm_client()
        parts = []
        for i, r in enumerate(results[:settings.synthesis_max_chunks], 1):
            p = f"--- Source {i} ---\nDoc: {r.file_name} | Type: {r.detected_doc_type}\n"
            p += f"Section: {r.heading_path or 'N/A'} | Score: {r.score:.3f}\n"
            p += f"Content:\n{r.chunk_text}\n"
            if r.chunk_keywords:
                p += f"Keywords: {', '.join(r.chunk_keywords)}\n"
            parts.append(p)
        prompt = RESPONSE_SYNTHESIS_PROMPT.format(query=query, results="\n".join(parts))
        try:
            return (await self._llm_client.complete(prompt, max_tokens=settings.synthesis_max_tokens,
                                                     temperature=settings.synthesis_temperature)).strip()
        except Exception as e:
            logger.error(f"Synthesis failed: {e}")
            return None

    def _sources(self, results):
        sm: dict[str, SourceInfo] = {}
        for r in results:
            if r.document_id not in sm:
                sm[r.document_id] = SourceInfo(document_id=r.document_id, file_name=r.file_name,
                    file_type=r.file_type, detected_doc_type=r.detected_doc_type,
                    chunks_used=1, chunk_ids=[r.chunk_id])
            else:
                sm[r.document_id].chunks_used += 1
                sm[r.document_id].chunk_ids.append(r.chunk_id)
        return list(sm.values())


response_generator = ResponseGenerator()


## Pipeline Orchestrator

In [ ]:
# ---------------------------------------------------------------------------
# Pipeline Orchestrator — end-to-end document processing for notebook use
# ---------------------------------------------------------------------------

class NotebookDocumentProcessor:
    """Process files: parse -> enrich doc -> chunk -> enrich chunks -> index."""

    async def process_file(self, file_path) -> dict:
        file_path = Path(file_path)
        if not file_path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")

        file_type = get_file_type(file_path)
        doc_id = hashlib.sha256(str(file_path.absolute()).encode()).hexdigest()[:16]
        file_hash = hashlib.sha256(file_path.read_bytes()).hexdigest()[:16]

        logger.info(f"Processing: {file_path.name}")

        # Parse
        pr = self._parse(file_path, file_type)
        if not pr.text:
            raise ValueError(f"No text from: {file_path}")

        # Doc enrichment
        enrichment = await entity_extractor.enrich(pr, file_path.name, file_type)

        # Store document
        doc = Document(id=doc_id, file_path=str(file_path.absolute()), file_name=file_path.name,
                       file_type=file_type, file_hash=file_hash,
                       detected_doc_type=enrichment.document_type, summary=enrichment.summary,
                       entities=enrichment.entities, key_topics=enrichment.key_topics,
                       table_descriptions=enrichment.table_descriptions,
                       indexed_at=datetime.utcnow(), processing_status="processing",
                       sheet_names=pr.sheet_names, column_schema=pr.column_types, row_count=pr.row_count)
        await metadata_store.add_document(doc)

        # Chunk
        tabular = is_tabular(file_type)
        if tabular:
            chunks = Chunker(chunk_size=settings.chunk_size).chunk_tabular(
                parse_result=pr, enrichment=enrichment, document_id=doc_id, file_name=file_path.name)
        elif settings.chunking_strategy == "paragraph":
            chunks = Chunker(chunk_size=settings.chunk_size).chunk_document(
                parse_result=ParseResult(text=pr.text, tables=[], headings=[]),
                enrichment=enrichment, document_id=doc_id, file_name=file_path.name)
        else:
            chunks = HierarchicalChunker(max_chunk_size=settings.max_chunk_size).chunk_document(
                markdown=pr.text, layout_data=None, document_id=doc_id,
                file_name=file_path.name, detected_doc_type=doc.detected_doc_type, entities=doc.entities)

        if not chunks:
            await metadata_store.update_document_status(doc_id, "failed")
            return {"status": "failed", "reason": "no chunks"}

        await metadata_store.add_chunks(chunks)
        await metadata_store.update_document_status(doc_id, "chunked")

        # Enrich chunks
        ctx = {"file_name": doc.file_name, "detected_doc_type": doc.detected_doc_type,
               "summary": doc.summary, "entities": doc.entities}
        try:
            er = await chunk_enricher.enrich_batch(chunks, ctx)
            nq = 0
            for (meta, qs), ch in zip(er, chunks):
                await metadata_store.add_chunk_metadata(meta)
                if qs:
                    await metadata_store.add_chunk_questions(ch.id, qs)
                    nq += len(qs)
            chunks = rebuild_contextualized_text(chunks, er, file_name=doc.file_name,
                                                  doc_type=doc.detected_doc_type, entities=doc.entities)
            await metadata_store.update_chunks_contextualized_text(chunks)
        except Exception as e:
            logger.warning(f"Chunk enrichment failed: {e}")

        # Index vectors
        await self._index(doc_id, chunks)

        logger.info(f"Done: {file_path.name} ({len(chunks)} chunks)")
        return {"document_id": doc_id, "file_name": file_path.name, "chunks": len(chunks),
                "doc_type": enrichment.document_type, "summary": enrichment.summary}

    async def search(self, query, **kw):
        return await response_generator.generate(query, **kw)

    def _parse(self, fp, ft):
        ext = fp.suffix.lower().lstrip(".")
        if ext == "csv":
            return CSVParser().parse(fp)
        if ext in ("xlsx", "xls"):
            return ExcelParser().parse(fp)
        if ext == "docx":
            return DocxProcessor().process(fp)
        if ext == "pptx":
            return PptxProcessor().process(fp)
        if ext == "pdf":
            return ParseResult(text=extract_pdf_text_simple(fp), tables=[], headings=[])
        try:
            return ParseResult(text=fp.read_text(encoding="utf-8", errors="replace"), tables=[], headings=[])
        except Exception:
            raise ValueError(f"Unsupported: {ft}")

    async def _index(self, doc_id, chunks):
        cids = [c.id for c in chunks]
        meta = await metadata_store.get_chunk_metadata_batch(cids)
        qmap = await metadata_store.get_chunk_questions_batch(cids)

        mt, mc, st, sc, qt, qm = [], [], [], [], [], []
        vecs = []

        for ch in chunks:
            m = meta.get(ch.id)
            qs = qmap.get(ch.id, [])

            mt.append(ch.contextualized_text)
            mc.append(ch.id)

            bm25_text = ch.text
            if m:
                parts = [p for p in [m.title, m.summary, " ".join(m.keywords) if m.keywords else None] if p]
                if parts:
                    bm25_text = " ".join(parts) + " " + bm25_text
            keyword_index.add(ch.id, bm25_text)

            vecs.append(VectorEmbedding(id=ch.id, chunk_id=ch.id, vector_type="main",
                                         source_text=ch.contextualized_text[:200]))
            if m and m.summary:
                st.append(m.summary)
                sc.append(ch.id)
                vecs.append(VectorEmbedding(id=f"{ch.id}_summary", chunk_id=ch.id,
                                             vector_type="summary", source_text=m.summary))
            for q in qs:
                qvid = f"q_{ch.id}_{q.id}"
                qt.append(q.question)
                qm.append((qvid, ch.id, q.id))
                vecs.append(VectorEmbedding(id=qvid, chunk_id=ch.id, vector_type="question",
                                             source_text=q.question, question_id=q.id))

        me = embedder.embed_batch(mt)
        se = embedder.embed_batch(st) if st else np.zeros((0, 0))
        qe = embedder.embed_batch(qt) if qt else np.zeros((0, 0))

        multi_vector_index.add_batch(
            main_data=list(zip(mc, me)),
            summary_data=list(zip(sc, se)) if st else None,
            question_data=[(q[0], q[1], qe[i]) for i, q in enumerate(qm)] if qt else None)

        for qvid, _, qid in qm:
            if qid:
                await metadata_store.update_question_vector_id(qid, qvid)

        await metadata_store.add_vector_embeddings_batch(vecs)
        await metadata_store.update_document_status(doc_id, "indexed")


processor = NotebookDocumentProcessor()


## Demo / Usage

In [ ]:
# ---------------------------------------------------------------------------
# Initialize all global instances
# ---------------------------------------------------------------------------

metadata_store = MetadataStore(":memory:")
await metadata_store.initialize()

multi_vector_index = MultiVectorIndex()
keyword_index = KeywordIndex()
enhanced_hybrid_search = EnhancedHybridSearch()

print("All components initialized!")
print(f"  Embedding model: {settings.embedding_model}")
print(f"  LLM provider: {settings.llm_provider}")
print(f"  Chunking strategy: {settings.chunking_strategy}")
print(f"  Database: in-memory SQLite")


In [ ]:
# ---------------------------------------------------------------------------
# Example: Process a document
# ---------------------------------------------------------------------------

# Uncomment and set path to process a file:
# result = await processor.process_file("path/to/your/document.pdf")
# print(result)

# Batch process:
# from pathlib import Path
# for f in Path("./documents").iterdir():
#     if f.suffix.lower() in ('.pdf', '.docx', '.pptx', '.csv', '.xlsx'):
#         try:
#             r = await processor.process_file(f)
#             print(f"  {f.name}: {r['chunks']} chunks")
#         except Exception as e:
#             print(f"  {f.name}: ERROR - {e}")

print("Uncomment the code above to process documents.")


In [ ]:
# ---------------------------------------------------------------------------
# Example: Search
# ---------------------------------------------------------------------------

# Uncomment after processing documents:
# response = await processor.search("What is the claims process?", mode="auto")
# print(f"Found {response.total_results} results in {response.latency_ms:.0f}ms")
# if response.answer:
#     print(f"\nAnswer:\n{response.answer}")
# for i, r in enumerate(response.results[:5], 1):
#     print(f"\n--- Result {i} (score={r.score:.3f}) ---")
#     print(f"  {r.file_name} | {r.heading_path or 'N/A'}")
#     print(f"  {r.chunk_text[:200]}...")

print("Uncomment the code above to search after processing documents.")
